In [1]:
using Distributed
addprocs(80)

80-element Vector{Int64}:
  2
  3
  4
  5
  6
  7
  8
  9
 10
 11
 12
 13
 14
  ⋮
 70
 71
 72
 73
 74
 75
 76
 77
 78
 79
 80
 81

In [23]:
rmprocs(2:81)

Task (done) @0x00007fa2e91ca9c0

In [13]:
procs()

1-element Vector{Int64}:
 1

In [2]:
using Distributed
using Arpack
@everywhere using WignerSymbols
@everywhere using LinearAlgebra
@everywhere using GenericLinearAlgebra
using BenchmarkTools
@everywhere using SparseArrays
using Plots
@everywhere using HDF5
@everywhere using H5Sparse
# using Arpack
@everywhere using Tullio
@everywhere using Octavian
@everywhere using ProgressMeter
@everywhere using SharedArrays
@everywhere using LoopVectorization
@everywhere using Quadmath
@everywhere using TSVD
@everywhere using JLD
#@everywhere using LowRankApprox
@everywhere push!(LOAD_PATH, ".")
@everywhere include("threej.jl")
@everywhere include("cuba.jl")
@everywhere include("Trans.jl")
@everywhere include("volume.jl")
@everywhere include("ME.jl")
@everywhere include("Sum_Functions.jl")
@everywhere floatabc=Float64

In [3]:
jm=[24 12 36]
cj1=spzeros((2*jm[1]+1)*(2*jm[2]+1)*(2*jm[3]+1), (2*jm[1]+1)*(2jm[2]+1))
@time tjsl2(jm,cj1)
aj1=cj1

2119.509138 seconds (505.10 M allocations: 20.739 GiB, 1.43% gc time, 0.01% compilation time)


89425×1225 SparseMatrixCSC{Float64, Int64} with 5195359 stored entries:
⎡⣷⡀⎤
⎢⣿⡅⎥
⎢⣿⡅⎥
⎢⣿⡆⎥
⎢⣿⡇⎥
⎢⣿⡇⎥
⎢⣿⣗⎥
⎢⣿⣯⎥
⎢⣿⣯⎥
⎢⣿⣷⎥
⎢⣿⡿⎥
⎢⣿⣿⎥
⎢⣿⣟⎥
⎢⣿⣿⎥
⎢⣿⣯⎥
⎢⣿⣷⎥
⎢⣿⡷⎥
⎢⣿⣿⎥
⎢⣿⣟⎥
⎢⣿⣿⎥
⎢⣿⣯⎥
⎢⣿⣿⎥
⎢⣿⣷⎥
⎢⣿⣿⎥
⎢⣛⣛⎥
⎣⣭⣭⎦

In [4]:
# 6-j Symbol
@everywhere jm61=[10 30 30 30 30]
@everywhere jm62=[30 10 30 30 30]
AA1=spzeros(Int((2*jm61[1]+1)*(2*jm61[2]+1)*(2*jm61[3]+1)*(2*jm61[4]+1)*(2*jm61[5]+1)))
sixj1(jm61,AA1)
@everywhere B1=$AA1
AA2=spzeros(Int((2*jm62[1]+1)*(2*jm62[2]+1)*(2*jm62[3]+1)*(2*jm62[4]+1)*(2*jm62[5]+1)))
sixj1(jm62,AA2)
@everywhere B2=$AA2

In [8]:
1

1

In [7]:
l=5
nl=25
res1=zeros(nl+4+1,l)*im
for i in 1:l
    ai1=i-1
    @everywhere t1=2.5
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]*10
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    ij=0
    th1=ij/180*pi
    z1v=im*[2 0 0]*10
    z2v=im*[0 2 0]*10
    z3v=im*[0 0 2]*10
    w1v=im*[cos(th1) sin(th1) 0]*20
    w2v=im*[0 2 0]*10
    w3v=im*[0 0 2]*10
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=i
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    @time tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=sqrt(abs(sum(at1[:,2]))*abs(sum(at1[:,3])))
    res1[4,i]=sum(at1[:,2])
    res1[5,i]=sum(at1[:,3])
    for jj in 5:nl+4
        res1[jj,i]=sum(at1[:,jj-2])
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

# save("~/julia code/data/res-new4-1.jld", "key14",res1)

  1.268230 seconds (2.46 k allocations: 405.906 KiB)
{2.5 + 0.0im,4.285315244611007e-216 + 0.0im,0.0 + 0.0im,3.294977468962207e-216 + 0.0im,3.294977468962207e-216 + 0.0im,-6.435502739323122e-216 + 0.0im,7.541604772644283e-215 + 0.0im,3.452272692555672e-213 + 0.0im,1.5803250240580772e-211 + 0.0im,7.234153857687733e-210 + 0.0im,3.3115328328038334e-208 + 0.0im,1.5158994290789036e-206 + 0.0im,6.939236888483749e-205 + 0.0im,3.176530558082772e-203 + 0.0im,1.4541002920911126e-201 + 0.0im,6.656342889821378e-200 + 0.0im,3.047031962503683e-198 + 0.0im,1.3948205394761731e-196 + 0.0im,6.3849817175728155e-195 + 0.0im,2.922812675890877e-193 + 0.0im,1.337957462718602e-191 + 0.0im,6.124683209466258e-190 + 0.0im,2.8036574750364348e-188 + 0.0im,1.2834125404524695e-186 + 0.0im,5.8749963704794444e-185 + 0.0im,2.6893599108029683e-183 + 0.0im,1.2310912677625895e-181 + 0.0im,5.6354885914425195e-180 + 0.0im,2.579721950428399e-178 + 0.0im},  3.362368 seconds (8.14 k allocations: 1.374 MiB)
{2.5 + 0.0im,3.73579

In [ ]:
nl=25
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(real(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

In [18]:
l=8
nl=25
res1=zeros(nl+4+1,l)*im
for i in 1:l
    ai1=i-1
    @everywhere t1=3-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    th1=30/180*pi
    th2=25/180*pi
    th3=40/180*pi
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    w1v=im*[cos(th1) sin(th1) 0]*6
    w2v=im*[-sin(th2) -cos(th2) 0]*6
    w3v=im*[0 cos(th3) sin(th3)]*6
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=16
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1女子自由式摔跤50公斤级
    希尔德布兰特
    for jj in 5:nl+4
        res1[jj,i]=sum(at1[:,jj-2])
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

# save("~/julia code/data/res-new4-2.jld", "key14",res1)

{3.0 + 0.0im,1.0296286234089371e-231 + 0.0im,0.0 + 0.0im,3.418134701612429e-229 + 0.0im,3.418134701612418e-229 + 0.0im,5.080559428561992e-231 - 5.059839506127566e-231im,-9.326736087288905e-230 - 5.102544276940727e-230im,1.288822526287672e-226 + 5.105191044429774e-227im,-3.3397127235295992e-223 - 5.627274972991489e-224im,1.205998149719438e-219 + 5.743306739528564e-221im,-5.949020863899512e-216 - 1.4317947611529579e-217im,4.0311631578080435e-212 + 1.6814938870054792e-213im,-3.603073521011312e-208 - 1.7381459749012114e-209im,4.002976317969248e-204 + 9.381625866825591e-206im,-5.309398319602488e-200 + 1.5327997350552732e-201im,8.296482307251556e-196 - 7.882379202564515e-197im,-1.53716991049951e-191 + 2.4187742668339702e-192im,3.4277434064751215e-187 - 6.917867388277151e-188im,-9.293835268413588e-183 + 2.1319175719344557e-183im,3.050361207977078e-178 - 7.860637935314698e-179im,-1.1893659971522939e-173 + 3.6360068078817333e-174im,5.376007366223926e-169 - 2.0386870708296978e-169im,-2.756005486

In [15]:
res1

30×8 Matrix{ComplexF64}:
           3.0+0.0im           …            1.6+0.0im
  8.92905e-231+0.0im                 9.228e-219+0.0im
           0.0+0.0im                        0.0+0.0im
  3.41813e-229+0.0im                9.6952e-215+0.0im
  3.41813e-229+0.0im                9.6952e-215+0.0im
  7.61439e-231-5.93518e-230im  …   7.10844e-219-5.54081e-218im
 -1.61247e-228+1.33067e-229im     -2.25916e-216-3.65766e-217im
  2.90805e-225-6.34342e-226im      3.97618e-213-2.21508e-214im
 -8.07571e-222+2.80574e-222im     -8.85087e-210+2.87162e-210im
  3.23132e-218-1.49872e-218im      2.42247e-206-1.46488e-206im
 -1.80095e-214+1.03517e-214im  …  -7.96583e-203+7.26062e-203im
  1.33926e-210-9.29951e-211im      3.08361e-199-3.9363e-199im
 -1.27232e-206+1.05574e-206im     -1.37523e-195+2.41135e-195im
              ⋮                ⋱  
 -5.89472e-181+1.05745e-180im      2.37319e-172+1.86397e-171im
  1.97137e-176-4.03623e-176im     -5.80754e-168-2.69501e-167im
 -7.35094e-172+1.76847e-171im  …   1.3107

In [10]:
nl=25
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(real(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{1.3,4.835692408108213e-213,0.0,1.1471155458592372e-207,1.1471155458592389e-207,4.428697498377024e-213,-1.5915563883159013e-210,2.9822690626987096e-207,-6.49754056498488e-204,1.6422889804641827e-200,-4.733840250054277e-197,1.52536576647905e-193,-5.330526837673603e-190,1.903817039321262e-186,-5.840378395756356e-183,4.0409989962581984e-181,3.0962573276735216e-175,-5.453912523789462e-171,8.015116958821752e-167,-1.1622911000427425e-162,1.7474040056775594e-158,-2.778321852172366e-154,4.7141283338243075e-150,-8.571461816704677e-146,1.6729379306455153e-141,-3.5061388276406797e-137,7.887167853506782e-133,-1.9027375243558307e-128,4.917142104461919e-124},

In [17]:
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(imag(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{3.0,0.0,0.0,0.0,0.0,-5.935182577311766e-230,1.3306742232645905e-229,-6.343416787711022e-226,2.805736351832155e-222,-1.4987170168828677e-218,1.0351675792375786e-214,-9.29950587391724e-211,1.0557387285055045e-206,-1.4681413390833896e-202,2.4525749926659218e-198,-4.879538853996847e-194,1.1524766853056373e-189,-3.220500764520925e-185,1.0574535560438352e-180,-4.0362285098467615e-176,1.7684726818222868e-171,-8.789758266328395e-167,4.90939909665932e-162,-3.06200055805207e-157,2.125215014096406e-152,-1.6392259163515058e-147,1.404715338745102e-142,-1.3367663789473918e-137,1.410295281884654e-132},{2.8,0.0,0.0,0.0,0.0,-5.8771418033051836e-229,8.205910280823659e-229,-6.215148951655164e-225,2.805198806367876e-221,-1.467564459264921e-217,9.755804469890599e-214,-8.210142532022313e-210,8.535859726116151e-206,-1.076975094230719e-201,1.634713391755178e-197,-2.9699885208867423e-193,6.416204863034592e-189,-1.6315961239174076e-184,4.823225048880559e-180,-1.6372182754338237e-175,6.317256104881773e-171,-2.7

In [7]:
l=1
nl=25
res1=zeros(nl+4+1,l)*im
for i in 1:l
    ai1=i-1
    @everywhere t1=1.5-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    th1=30/180*pi
    th2=60/180*pi
    th3=80/180*pi
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    w1v=im*[cos(th1) sin(th1) 0]*6
    w2v=im*[sin(th2) cos(th2) 0]*6
    w3v=im*[0 cos(th3) sin(th3)]*6
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=16
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=sqrt(abs(sum(at1[:,2]))*abs(sum(at1[:,3])))
    res1[4,i]=sum(at1[:,2])
    res1[5,i]=sum(at1[:,3])
    for jj in 5:nl+4
        res1[jj,i]=sum(at1[:,jj-2])
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

# save("~/julia code/data/res-new4-3.jld", "key14",res1)

{1.5 + 0.0im,8.675740361706792e-93 + 0.0im,1.0825281251435896e-92 + 0.0im,1.0825281251435911e-92 + 0.0im,1.0825281251435879e-92 + 0.0im,-1.966173195503628e-93 - 2.4949529784891614e-92im,-9.310966625847203e-91 + 1.1171122358825883e-91im,1.804508122671307e-87 - 3.407546985412641e-88im,-4.850060739568831e-84 + 1.1782409238925945e-84im,1.6836479197648965e-80 - 4.850084609383826e-81im,-7.269488572896316e-77 + 2.377376050697765e-77im,3.80825582620667e-73 - 1.3768951010689525e-73im,-2.376842214902563e-69 + 9.33576835591133e-70im,1.742303754234628e-65 - 7.34350506801519e-66im,-1.4826101527617673e-61 + 6.6452267213089935e-62im,1.4502761836159849e-57 - 6.86517539840405e-58im,-1.6171334974546588e-53 + 8.041306109490352e-54im,2.0404958603305674e-49 - 1.0612522069277379e-49im,-2.894809570510588e-45 + 1.5691635886344146e-45im,4.5909924755588766e-41 - 2.586085437717735e-41im,-8.097676958386429e-37 + 4.728334954384166e-37im,1.5811150554582236e-32 - 9.549989601104814e-33im,-3.4031555264619856e-28 + 2.1

In [ ]:
l=1
nl=25
res1=zeros(nl+4+1,l)*im
for i in 1:l
    ai1=i-1
    @everywhere t1=1.5-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    th1=10/180*pi
    th2=60/180*pi
    th3=80/180*pi
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    w1v=im*[cos(th1) sin(th1) 0]*6
    w2v=im*[sin(th2) cos(th2) 0]*6
    w3v=im*[0 cos(th3) sin(th3)]*6
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=17
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=sqrt(abs(sum(at1[:,2]))*abs(sum(at1[:,3])))
    res1[4,i]=sum(at1[:,2])
    res1[5,i]=sum(at1[:,3])
    for jj in 5:nl+4
        res1[jj,i]=sum(at1[:,jj-2])
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

In [18]:
nl=25
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(real(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{1.5,7.274805552942917e-98,1.0825281251435913e-92,1.0825281251435911e-92,1.0825281251435914e-92,4.13005778547507e-108,-2.832627482033135e-96,6.77195430842578e-94,1.0931515690438907e-90,-4.6804235389888554e-87,1.534048394769879e-83,-5.24546546405935e-80,1.9626762524852173e-76,-8.027071779631652e-73,3.504361192986229e-69,-1.5410291439453778e-65,5.648278887450742e-62,3.477384656265985e-59,-5.439814042378803e-54,1.163318379404046e-49,-2.1427613128173084e-45,3.946939331510902e-41,-7.60243562071132e-37,1.5578336974202603e-32,-3.419677512273997e-28,8.061232954948355e-24,-2.041196633957348e-19,5.547342433561277e-15,-1.6159555753003578e-10},

In [19]:
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(imag(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{1.5,0.0,0.0,0.0,0.0,-1.1539147978903941e-97,-1.0729933221205303e-96,1.719636441866593e-93,-1.967255934340142e-90,1.5893381855720628e-87,2.2961254039320617e-84,-2.5860084631056448e-80,1.6266799747004257e-76,-1.0033620590873734e-72,6.673553869504913e-69,-4.919512405823982e-65,4.0459503359689595e-61,-3.707755258954604e-57,3.770968800827989e-53,-4.236703780359048e-49,5.234260231151698e-45,-7.07970938678473e-41,1.0436404176102537e-36,-1.6684324286423808e-32,2.8756048206888395e-28,-5.303008273613019e-24,1.0353050826081117e-19,-2.1043962680567582e-15,4.320481113452262e-11},

In [5]:
l=3
nl=25
res1=zeros(nl+4+1,l)*im
for i in 1:l
    ai1=i-1
    @everywhere t1=1.5-0.1*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    th1=150/180*pi
    th2=0/180*pi
    th3=90/180*pi
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    w1v=im*[cos(th1) sin(th1) 0]*6
    w2v=im*[sin(th2) cos(th2) 0]*6
    w3v=im*[0 cos(th3) sin(th3)]*6
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=17
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=sqrt(abs(sum(at1[:,2]))*abs(sum(at1[:,3])))
    res1[4,i]=sum(at1[:,2])
    res1[5,i]=sum(at1[:,3])
    for jj in 5:nl+4
        res1[jj,i]=sum(at1[:,jj-2])
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end


{1.5 + 0.0im,7.274805552942919e-98 + 0.0im,1.0825281251435916e-92 + 0.0im,1.0825281251435914e-92 + 0.0im,1.0825281251435916e-92 + 0.0im,4.130057785475071e-108 - 1.1539147978903942e-97im,-2.8326274820331373e-96 - 1.0729933221205305e-96im,6.771954308425835e-94 + 1.7196364418665947e-93im,1.0931515690438604e-90 - 1.9672559343401605e-90im,-4.680423538988643e-87 + 1.58933818557224e-87im,1.5340483947697063e-83 + 2.296125403930121e-84im,-5.245465464057798e-80 - 2.586008463103256e-80im,1.96267625248374e-76 + 1.62667997469716e-76im,-8.027071779618089e-73 - 1.0033620590824494e-72im,3.5043611929776393e-69 + 6.673553869423719e-69im,-1.5410291439551188e-65 - 4.919512405678802e-65im,5.648278888159427e-62 + 4.045950335690503e-61im,3.477384629527899e-59 - 3.707755258389536e-57im,-5.439814033394783e-54 + 3.7709687996382287e-53im,1.163318376445577e-49 - 4.236703777843618e-49im,-2.1427613029674305e-45 + 5.234260226177489e-45im,3.946939298003537e-41 - 7.079709379610101e-41im,-7.60243550418461e-37 + 1.04364

In [6]:
nl=25
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(real(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end


{1.5,7.274805552942919e-98,1.0825281251435916e-92,1.0825281251435914e-92,1.0825281251435916e-92,4.130057785475071e-108,-2.8326274820331373e-96,6.771954308425835e-94,1.0931515690438604e-90,-4.680423538988643e-87,1.5340483947697063e-83,-5.245465464057798e-80,1.96267625248374e-76,-8.027071779618089e-73,3.5043611929776393e-69,-1.5410291439551188e-65,5.648278888159427e-62,3.477384629527899e-59,-5.439814033394783e-54,1.163318376445577e-49,-2.1427613029674305e-45,3.946939298003537e-41,-7.60243550418461e-37,1.5578336562930133e-32,-3.4196773675291893e-28,8.061232466509615e-24,-2.0411964921116962e-19,5.54734224093089e-15,-1.615955782716077e-10},{1.4,4.748348653036229e-96,2.345395491455037e-90,2.345395491455037e-90,2.345395491455037e-90,3.632475696274902e-106,-1.9600293869319413e-94,5.4823079651554605e-92,6.494435223599415e-89,-2.942956295545371e-85,9.257849589861346e-82,-2.947066934741424e-78,1.0055894333835117e-74,-3.676648155303919e-71,1.3917618794184873e-67,-4.891758910792576e-64,8.3799415293

In [7]:
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(imag(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{1.5,0.0,0.0,0.0,0.0,-1.1539147978903942e-97,-1.0729933221205305e-96,1.7196364418665947e-93,-1.9672559343401605e-90,1.58933818557224e-87,2.296125403930121e-84,-2.586008463103256e-80,1.62667997469716e-76,-1.0033620590824494e-72,6.673553869423719e-69,-4.919512405678802e-65,4.045950335690503e-61,-3.707755258389536e-57,3.7709687996382287e-53,-4.236703777843618e-49,5.234260226177489e-45,-7.079709379610101e-41,1.0436404183665322e-36,-1.6684324438760979e-32,2.875604930774536e-28,-5.303008944592377e-24,1.0353054715533093e-19,-2.1043984979510844e-15,4.320493956246274e-11},{1.4,0.0,0.0,0.0,0.0,-7.930893550707843e-96,-6.55731448367725e-95,1.1526310258729265e-91,-1.276712896210404e-88,9.447913454754198e-86,1.59947830933724e-82,-1.5648430142846826e-78,9.174948410912811e-75,-5.287416853253149e-71,3.2662808558076076e-67,-2.2163326524273826e-63,1.661882029627993e-59,-1.3765088155479518e-55,1.2560808280441995e-51,-1.2582977097424434e-47,1.3781336339348969e-43,-1.642198726079872e-39,2.1161326427748658e-

In [8]:
l=3
nl=25
res1=zeros(nl+4+1,l)*im
for i in 1:l
    ai1=i-1
    @everywhere t1=1.2-0.1*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]*10^(-40)
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    th1=150/180*pi
    th2=0/180*pi
    th3=90/180*pi
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    w1v=im*[cos(th1) sin(th1) 0]*6
    w2v=im*[sin(th2) cos(th2) 0]*6
    w3v=im*[0 cos(th3) sin(th3)]*6
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=20
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=sqrt(abs(sum(at1[:,2]))*abs(sum(at1[:,3])))
    res1[4,i]=sum(at1[:,2])
    res1[5,i]=sum(at1[:,3])
    for jj in 5:nl+4
        res1[jj,i]=sum(at1[:,jj-2])
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

# save("~/julia code/data/res-new4-3.jld", "key14",res1)

{1.2 + 0.0im,1.3420731927127953e-91 + 0.0im,1.5492398621953099e-84 + 0.0im,1.5492398621953096e-84 + 0.0im,1.54923986219531e-84 + 0.0im,-9.637999322284523e-101 - 2.9259474238311913e-91im,-7.334845312349199e-90 - 1.850140462210728e-90im,2.79062204184506e-87 + 3.96991829695214e-87im,1.6843900350418218e-84 - 4.357286828541747e-84im,-9.14223638489652e-81 + 2.9977315384150755e-81im,2.6305444054772462e-77 + 5.361151674304996e-78im,-7.194381806294294e-74 - 4.541235227418896e-74im,2.013470363860712e-70 + 2.336176151229913e-70im,-5.648521970289378e-67 - 1.1554845950173463e-66im,1.3776408672381815e-63 + 5.981872567587552e-63im,-5.572832288536044e-61 - 3.3286979722459384e-59im,-3.696310795095535e-56 + 2.0086439079489464e-55im,5.166853447996132e-52 - 1.3160136370607024e-51im,-5.855768193721716e-48 + 9.33076204470516e-48im,6.486727104393737e-44 - 7.107226278070636e-44im,-7.40222937002387e-40 + 5.743584364088749e-40im,8.882374887436368e-36 - 4.8190922607106516e-36im,-1.1310698769038188e-31 + 4.021617

In [ ]:
nl=25
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(real(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{1.1,8.982293824530404e-92,7.6017501102166445e-81,7.601750110216764e-81,7.601750110216524e-81,6.376221157694727e-92,-1.6330316700117667e-90,1.633988759596176e-88,3.223032989023817e-85,-1.0590408313021004e-81,2.7642095119519637e-78,-7.345288679127033e-75,2.0971831695403726e-71,-6.467270568975995e-68,2.088339351320552e-64,-6.284831535699992e-61,7.66357693512053e-58,1.814078561416345e-53,-3.578997914201021e-49,5.475451610377072e-45,-8.228538264095716e-41,1.2843320076896202e-36,-2.1201774659926345e-32,3.7261031454505265e-28,-6.9896586631991e-24,1.4012819445571744e-19,-3.004610671025152e-15,6.892790161263342e-11,-1.6915441684665533e-6},{1.0,1.0646372978940765e-88,1.9887984467032967e-76,1.988798446703332e-76,1.9887984467032614e-76,8.047241573467121e-89,-2.048387014121523e-87,1.8414511868885227e-85,3.335806228401211e-82,-1.042779224992677e-78,2.3752792093999405e-75,-4.521177509348807e-72,3.2122417875557065e-69,4.925083079977318e-65,-5.745766122813627e-61,5.219850906925988e-57,-4.6605126790271

In [ ]:
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(imag(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{1.1,0.0,0.0,0.0,0.0,-1.2250042449211429e-91,-3.8390255376815565e-91,-6.694970854187859e-88,1.1776928428563752e-84,-1.8172867545730526e-81,3.0142505279095593e-78,-5.480283426029917e-75,1.084236865326052e-71,-2.3117107424971232e-68,5.430712847476265e-65,-1.6693745898032313e-61,9.538676647210824e-58,-9.454236646484766e-54,1.1374680718860104e-49,-1.448114948393787e-45,1.9167621386590145e-41,-2.6574313351497507e-37,3.902004705995758e-33,-6.140926055101417e-29,1.0490507197374209e-24,-1.9682949381962414e-20,4.0870666326506067e-16,-9.394466938784825e-12,2.3727580711578363e-7},{1.0,0.0,0.0,0.0,0.0,-1.5442706583391458e-88,-7.488635381254343e-88,-6.071268674867282e-85,1.1939047885717066e-81,-1.7969916255381037e-78,2.907776076417357e-75,-5.567228425532336e-72,1.4274086835753458e-68,-5.504769924707292e-65,3.0722817688861506e-61,-2.1606323101902917e-57,1.7401264083678804e-53,-1.5409813821193714e-49,1.4771590075323437e-45,-1.521927635169556e-41,1.6779418420033636e-37,-1.9718984653833084e-33,2.458849

In [9]:
l=1
nl=25
res1=zeros(nl+4+1,l)*im
for i in 1:l
    ai1=i-1
    @everywhere t1=0.9-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    th1=120/180*pi
    th2=0/180*pi
    th3=90/180*pi
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    w1v=im*[cos(th1) sin(th1) 0]*6
    w2v=im*[sin(th2) cos(th2) 0]*6
    w3v=im*[0 cos(th3) sin(th3)]*6
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=22
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=sqrt(abs(sum(at1[:,2]))*abs(sum(at1[:,3])))
    res1[4,i]=sum(at1[:,2])
    res1[5,i]=sum(at1[:,3])
    for jj in 5:nl+4
        res1[jj,i]=sum(at1[:,jj-2])
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

In [ ]:
nl=25
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(real(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{0.8,1.054331888747649e-70,2.4859515981238238e-64,2.4859515981238404e-64,2.4859515981238073e-64,3.17128442200207e-80,-1.1483392565642821e-68,1.7636533762154738e-65,-3.0826944151680167e-62,6.055481062644666e-59,-1.326486567565184e-55,3.212753957419678e-52,-8.512098256636297e-49,2.4319095243910487e-45,-7.331846115131328e-42,2.2433373146659567e-38,-6.342713471876998e-35,1.087710703431591e-31,6.2449804361636825e-28,-1.1464610465591341e-23,1.2743371703078377e-19,-1.2882032274098615e-15,1.2906238043814017e-11,-1.3228745583242952e-7,0.0014074173021613517,-15.654994981407958,182728.07795971644,-2.241958663573055e9,2.8930637493211637e13},{0.8,0.0,0.0,0.0,0.0,-3.145733189835087e-70,-3.13913127187727e-70,2.3540108330554238e-66,-8.663667522541571e-63,2.747290667724603e-59,-8.656131778028242e-56,2.8557613474317267e-52,-1.008719161531325e-48,3.852128232704985e-45,-1.5965208307053543e-41,7.186788639699629e-38,-3.5108416582988236e-34,1.8579414674519206e-30,-1.0626513914874392e-26,6.551407731955525e-23

In [ ]:
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(imag(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

In [ ]:
l=1
nl=25
res1=zeros(nl+4+1,l)*im
for i in 1:l
    ai1=i-1
    @everywhere t1=0.95-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    th1=120/180*pi
    th2=0/180*pi
    th3=90/180*pi
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    w1v=im*[cos(th1) sin(th1) 0]*6
    w2v=im*[sin(th2) cos(th2) 0]*6
    w3v=im*[0 cos(th3) sin(th3)]*6
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=21
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=sqrt(abs(sum(at1[:,2]))*abs(sum(at1[:,3])))
    res1[4,i]=sum(at1[:,2])
    res1[5,i]=sum(at1[:,3])
    for jj in 5:nl+4
        res1[jj,i]=sum(at1[:,jj-2])
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

In [ ]:
nl=25
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(real(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

In [5]:
l=1
nl=25
res1=zeros(nl+4+1,l)*im
for i in 1:l
    ai1=i-1
    @everywhere t1=0.8-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    th1=120/180*pi
    th2=0/180*pi
    th3=90/180*pi
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    w1v=im*[cos(th1) sin(th1) 0]*6
    w2v=im*[sin(th2) cos(th2) 0]*6
    w3v=im*[0 cos(th3) sin(th3)]*6
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=23
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=sqrt(abs(sum(at1[:,2]))*abs(sum(at1[:,3])))
    res1[4,i]=sum(at1[:,2])
    res1[5,i]=sum(at1[:,3])
    for jj in 5:nl+4
        res1[jj,i]=sum(at1[:,jj-2])
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

{0.8 + 0.0im,1.054331888747649e-70 + 0.0im,2.4859515981238238e-64 + 0.0im,2.4859515981238404e-64 + 0.0im,2.4859515981238073e-64 + 0.0im,3.17128442200207e-80 - 3.145733189835087e-70im,-1.1483392565642821e-68 - 3.13913127187727e-70im,1.7636533762154738e-65 + 2.3540108330554238e-66im,-3.0826944151680167e-62 - 8.663667522541571e-63im,6.055481062644666e-59 + 2.747290667724603e-59im,-1.326486567565184e-55 - 8.656131778028242e-56im,3.212753957419678e-52 + 2.8557613474317267e-52im,-8.512098256636297e-49 - 1.008719161531325e-48im,2.4319095243910487e-45 + 3.852128232704985e-45im,-7.331846115131328e-42 - 1.5965208307053543e-41im,2.2433373146659567e-38 + 7.186788639699629e-38im,-6.342713471876998e-35 - 3.5108416582988236e-34im,1.087710703431591e-31 + 1.8579414674519206e-30im,6.2449804361636825e-28 - 1.0626513914874392e-26im,-1.1464610465591341e-23 + 6.551407731955525e-23im,1.2743371703078377e-19 - 4.341160929595709e-19im,-1.2882032274098615e-15 + 3.0821479695236473e-15im,1.2906238043814017e-11 - 2

In [6]:
nl=25
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(real(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{0.8,1.054331888747649e-70,2.4859515981238238e-64,2.4859515981238404e-64,2.4859515981238073e-64,3.17128442200207e-80,-1.1483392565642821e-68,1.7636533762154738e-65,-3.0826944151680167e-62,6.055481062644666e-59,-1.326486567565184e-55,3.212753957419678e-52,-8.512098256636297e-49,2.4319095243910487e-45,-7.331846115131328e-42,2.2433373146659567e-38,-6.342713471876998e-35,1.087710703431591e-31,6.2449804361636825e-28,-1.1464610465591341e-23,1.2743371703078377e-19,-1.2882032274098615e-15,1.2906238043814017e-11,-1.3228745583242952e-7,0.0014074173021613517,-15.654994981407958,182728.07795971644,-2.241958663573055e9,2.8930637493211637e13},

In [7]:
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(imag(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{0.8,0.0,0.0,0.0,0.0,-3.145733189835087e-70,-3.13913127187727e-70,2.3540108330554238e-66,-8.663667522541571e-63,2.747290667724603e-59,-8.656131778028242e-56,2.8557613474317267e-52,-1.008719161531325e-48,3.852128232704985e-45,-1.5965208307053543e-41,7.186788639699629e-38,-3.5108416582988236e-34,1.8579414674519206e-30,-1.0626513914874392e-26,6.551407731955525e-23,-4.341160929595709e-19,3.0821479695236473e-15,-2.3367861300993627e-11,1.8849369385765108e-7,-0.0016108705428861812,14.512254616761767,-136952.03520248452,1.3421769483490534e9,-1.34843521397267e13},

In [39]:
l=20
nl=25
res1=zeros(nl+4+1,l)*im
    ai1=1-1
    @everywhere t1=2.5-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    th1=150/180*pi
    th2=0/180*pi
    th3=90/180*pi
    z1v=im*[2 0 0]*10
    z2v=im*[0 2 0]*10
    z3v=im*[0 0 2]*10
    w1v=im*[cos(th1) sin(th1) 0]*10
    w2v=im*[sin(th2) cos(th2) 0]*10
    w3v=im*[0 cos(th3) sin(th3)]*10
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}(l,nl+4)
    for i in 1:l
            jn=i
            js1=jn
            js2=jn
            js3=jn
            js4=js1+js3+js3
        tran123gv3j(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
        res1[1,i]=abs(t1)
        res1[2,i]=abs(sum(at1[i,1]))
        res1[3,i]=sqrt(abs(sum(at1[i,2]))*abs(sum(at1[i,3])))
        res1[4,i]=sum(at1[i,2])
        res1[5,i]=sum(at1[i,3])
        for jj in 5:nl+4
            res1[jj,i]=sum(at1[i,jj-2])
        end
    end

In [37]:
res1[3,:]

20-element Vector{ComplexF64}:
   4.604870910244036e24 + 0.0im
   4.257889406801786e47 + 0.0im
   4.223780245079786e68 + 0.0im
   6.804386756732926e87 + 0.0im
 2.0703461984717975e105 + 0.0im
  1.280548786753111e121 + 0.0im
  1.678528675671261e135 + 0.0im
  4.785271204594796e147 + 0.0im
                    Inf + 0.0im
                    Inf + 0.0im
                    Inf + 0.0im
                    Inf + 0.0im
                    Inf + 0.0im
                    Inf + 0.0im
                    Inf + 0.0im
                    Inf + 0.0im
                    Inf + 0.0im
                    Inf + 0.0im
                    Inf + 0.0im
                    Inf + 0.0im

In [40]:
res1[3,:]

20-element Vector{ComplexF64}:
  4.604870910244033e-96 + 0.0im
  4.257889406801784e-73 + 0.0im
 4.2237802450797834e-52 + 0.0im
  6.804386756732926e-33 + 0.0im
 2.0703461984717982e-15 + 0.0im
     12.805487867531108 + 0.0im
   1.678528675671259e15 + 0.0im
  4.7852712045947943e27 + 0.0im
  3.0188101959236213e38 + 0.0im
  4.2655377391153116e47 + 0.0im
  1.3619083457244027e55 + 0.0im
   9.890815854557848e60 + 0.0im
   1.642253112593047e65 + 0.0im
   6.259119591052647e67 + 0.0im
   5.493465595990497e68 + 0.0im
  1.1132024590052442e68 + 0.0im
   5.219535497833067e65 + 0.0im
   5.672829761770726e61 + 0.0im
  1.4313159264827186e56 + 0.0im
   8.394573015498753e48 + 0.0im

In [45]:
nl=25
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(real(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{2.5,4.285315244611006e-96,4.604870910244033e-96,3.2949774025334364e-96,-6.435502739323121e-96,7.541604772644283e-95,3.452272692555673e-93,1.5803250240580787e-91,7.234153857687741e-90,3.3115328328038377e-88,1.515899429078906e-86,6.939236888483765e-85,3.176530558082779e-83,1.4541002920911165e-81,6.656342889821399e-80,3.0470319625036936e-78,1.3948205394761785e-76,6.384981717572842e-75,2.92281267589089e-73,1.337957462718608e-71,6.124683209466288e-70,2.803657475036449e-68,1.2834125404524764e-66,5.8749963704794795e-65,2.6893599108029853e-63,1.2310912677625977e-61,5.635488591442558e-60,2.579721950428418e-58,1.18090299244465e-56},{2.5,3.735798398878354e-73,4.257889406801784e-73,1.0771702840477498e-73,-1.6830785688246092e-72,5.259620527576902e-71,3.8522611285963635e-68,2.8214803187961647e-65,2.0665139053682845e-62,1.5135599892834113e-59,1.108564445275936e-56,8.119368495673359e-54,5.946803097417012e-51,4.355568674865973e-48,3.190113775536603e-45,2.3365091129418467e-42,1.7113103854554542e-39,1.2

In [44]:
res1[4,15]

6.766194860008656e66 + 0.0im

In [43]:
res1[4,:]

20-element Vector{ComplexF64}:
 3.2949774025334364e-96 + 0.0im
 1.0771702840477498e-73 + 0.0im
  5.816401972349825e-53 + 0.0im
  6.086028533301659e-34 + 0.0im
 1.3250215670219505e-16 + 0.0im
     0.6234537391027122 + 0.0im
   6.485100123096627e13 + 0.0im
  1.5132356228137147e26 + 0.0im
   8.000313848005545e36 + 0.0im
   9.651810114353819e45 + 0.0im
  2.6711226745452598e53 + 0.0im
  1.7025317815603043e59 + 0.0im
  2.5070329716772923e63 + 0.0im
   8.549804347967431e65 + 0.0im
   6.766194860008656e66 + 0.0im
  1.2445981855278252e66 + 0.0im
   5.328358706567778e63 + 0.0im
  5.3152778742476615e59 + 0.0im
  1.2366308678450334e54 + 0.0im
   6.715658412399016e46 + 0.0im

In [46]:
ii=0
jj=0
nt=20
nl=12
res1=Array{Float64,2}(undef,nl+1+4+2,nt)
        ai1=0
        @everywhere t1=1-0.05*$ai1
#         println("t1=",t1)
        A=pi - asin(2*sqrt(2)/3);
        A1=A+ii/15
        A2=A-jj/15
        jp4=2*[1 1 1 1]*3
        # z1v=im*pv(jp4,A)[:,1]
        # z2v=im*pv(jp4,A)[:,2]
        # z3v=im*pv(jp4,A)[:,3]
        # z4v=im*pv(jp4,A)[:,4]
        z1v=im*pv2(jp4,A1,A2)[1][:,1]
        z2v=im*pv2(jp4,A1,A2)[1][:,2]
        z3v=im*pv2(jp4,A1,A2)[1][:,3]
        z4v=im*pv2(jp4,A1,A2)[1][:,4]
        @everywhere g4f1=slce2($z1v)
        @everywhere g4f2=slce2($z2v)
        @everywhere g4f3=slce2($z3v)
        @everywhere g4f4=slce2($z4v)
        nn=0
        k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
        g4tf1=transpose(conj(g4f1))
        g4tf2=transpose(conj(g4f2))
        g4tf3=transpose(conj(g4f3))
        g4tf4=transpose(conj(g4f4))
#         fr1=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#         fr2=(x)->itfro([x[1] x[2] x[3]],h4tf1,h4tf2,h4tf3,h4tf4,h4f1,h4f2,h4f3,h4f4,nn,t1,k)
#         (val,err) = hcubature(fr1, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         (val2,err2) = hcubature(fr2, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
#         val22=val2*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}(nt,nl+4)
    for i in 1:nt
        jn=i
        js1=jn
        js2=jn
        js3=jn
        js4=jn
        tran123test_sj(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,jm,aj1,5/t1,100,at1,nl)
        res1[1,i]=abs(t1)
        res1[2,i]=A1
        res1[3,i]=A2
        res1[4,i]=abs(sum(at1[i,1]))
        res1[5,i]=abs(sum(at1[i,2]))
        for jj in 6:nl+1+4+2
            res1[jj,i]=abs(sum(at1[i,jj-3]))
        end
    end

In [47]:
nl=12
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(real(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{1.0,1.9106332362490182,1.9106332362490182,9.697574741980835e8,1.0420722229344049e9,4.5120824758819216e8,1.9538854180020097e8,3.6635351587537706e7,6.869128422663323e6,1.2879615792493736e6,241492.79610925767,45279.89927048584,8489.981113216098,1591.8714587280192,298.4758985115037,0.0},{1.0,1.9106332362490182,1.9106332362490182,3.4616054127058854e17,1.8598978545208877e17,3.2213868158290426e17,5.57960563849705e17,1.6738816915491151e18,5.021645074647345e18,1.5064935223942033e19,4.5194805671826104e19,1.3558441701547832e20,4.067532510464349e20,1.220259753139305e21,3.660779259417914e21,0.0},{1.0,1.9106332362490182,1.9106332362490182,1.4747331998463493e25,4.950545624784661e24,2.178206790497579e25,9.746286074343171e25,1.9187983728486902e27,3.7776340100020106e28,7.437216908837182e29,1.464202078111342e31,2.8826478411440077e32,5.675212937229029e33,1.1173075470165725e35,2.1996992331888112e36,0.0},{1.0,1.9106332362490182,1.9106332362490182,7.765257125284023e31,1.8195135743500667e31,1.651194085172708

In [48]:
res1[5,:]

20-element Vector{Float64}:
 1.0420722229344049e9
 1.8598978545208877e17
 4.950545624784661e24
 1.8195135743500667e31
 9.03029965368768e36
 6.0196758103228105e41
 5.385918080758564e45
 6.471335543861682e48
 1.0450347514879356e51
 2.2699937946920409e52
 6.6373439810124e52
 2.614061218515887e52
 1.387474679808316e51
 9.929413327506286e48
 9.584804217638959e45
 1.2483931927089333e42
 2.1945976781989287e37
 5.208391735388833e31
 1.6691413524473465e25
 7.224485011566954e17

In [49]:
res1[5,12]

2.614061218515887e52

In [50]:
sqrt(5.5*(5.5+1))

5.979130371550699

In [51]:
sqrt(7.5*(7.5+1))

7.984359711335656

In [ ]:
l=1
nl=25
res1=zeros(nl+4+1,l)*im
for i in 1:l
    ai1=i-1
    @everywhere t1=1-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    ij=10
    th1=ij/180*pi
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    w1v=im*[cos(th1) sin(th1) 0]*6
    w2v=im*[0 2 0]*3
    w3v=im*[0 0 2]*3
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=21
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=sqrt(abs(sum(at1[:,2]))*abs(sum(at1[:,3])))
    res1[4,i]=sum(at1[:,2])
    res1[5,i]=sum(at1[:,3])
    for jj in 5:nl+4
        res1[jj,i]=sum(at1[:,jj-2])
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/res-new4-4.jld", "key14",res1)

In [ ]:
l=1
nl=25
res1=zeros(nl+4+1,l)*im
for i in 1:l
    ai1=i-1
    @everywhere t1=0.8-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    ij=10
    th1=ij/180*pi
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    w1v=im*[cos(th1) sin(th1) 0]*6
    w2v=im*[0 2 0]*3
    w3v=im*[0 0 2]*3
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=23
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=sqrt(abs(sum(at1[:,2]))*abs(sum(at1[:,3])))
    res1[4,i]=sum(at1[:,2])
    res1[5,i]=sum(at1[:,3])
    for jj in 5:nl+4
        res1[jj,i]=sum(at1[:,jj-2])
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/res-new4-5.jld", "key14",res1)

In [ ]:
l=1
nl=25
res1=Array{Float64,2}(undef,nl+4+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=0.8-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    ij=10
    th1=ij/180*pi
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    w1v=im*[cos(th1) sin(th1) 0]*6
    w2v=im*[0 2 0]*3
    w3v=im*[0 0 2]*3
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=21
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=sqrt(abs(sum(at1[:,2]))*abs(sum(at1[:,3])))
    res1[4,i]=abs(sum(at1[:,2]))
    res1[5,i]=abs(sum(at1[:,3]))
    for jj in 5:nl+4
        res1[jj,i]=abs(sum(at1[:,jj-2]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/res-new4-4-2.jld", "key14",res1)

In [ ]:
l=15
nl=30
res1=Array{Float64,2}(undef,nl+4+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=4.4-0.1*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[2 0 0]*10
    z2v=im*[0 2 0]*10
    z3v=im*[0 0 2]*10
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    nn=0
    jn=18
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result310n.jld", "key310",res1)

In [ ]:
l=20
nl=30
res1=Array{Float64,2}(undef,nl+4+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=1.6-0.02*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    nn=0
    jn=17
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result2-1.jld", "key1",res1)

In [ ]:
l=5
nl=30
res1=Array{Float64,2}(undef,nl+4+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=1.2-0.02*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[2 0 0]*3
    z2v=im*[0 2 0]*3
    z3v=im*[0 0 2]*3
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    nn=0
    jn=19
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result2-2.jld", "key1",res1)

In [ ]:
l=5
nl=30
res1=Array{Float64,2}(undef,nl+4+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=1.4-0.02*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[2 0 0]*5
    z2v=im*[0 2 0]*5
    z3v=im*[0 0 2]*5
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    nn=0
    jn=18
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result2.jld", "key1",res1)

In [ ]:
l=6
nl=30
res1=Array{Float64,2}(undef,nl+4+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=1.6-0.02*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[2 0 0]*5
    z2v=im*[0 2 0]*5
    z3v=im*[0 0 2]*5
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    nn=0
    jn=20
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result3.jld", "key1",res1)

In [6]:
nl=25
res1=load("~/julia code/data/res-new4-5.jld", "key14")
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(real(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{0.8,2.9882993383806216e-183,0.0,2.4859515981238414e-184,2.485951598123805e-184,-4.0570068312369476e-182,7.764493594919825e-180,3.2420356796967123e-175,1.5761065516998558e-170,8.768978134763014e-166,5.5178886870324065e-161,3.8917176714096745e-156,3.0539753866426504e-151,2.649881637624964e-146,2.5282651116657973e-141,2.6392021631330005e-136,3.000198932893208e-131,3.69775329193899e-126,4.920381606343415e-121,7.03971881411776e-116,1.0786488626707752e-110,1.763161183787622e-105,3.063074785779668e-100,5.635017670846314e-95,1.0939088371397279e-89,2.233338459330888e-84,4.779945496994166e-79,1.069214470011148e-73,2.4924756700267902e-68},

In [12]:
res1=load("~/julia code/data/res-new4-5.jld", "key14")
for i in 1:size(res1)[2]
    str1="{"*string(real(res1[1,i]))
    for kk in 2:nl+4
        str1=str1*","*string(imag(res1[kk,i]))
    end
    str1=str1*"},"
    print(str1)
end

{0.8,0.0,0.0,0.0,0.0,-7.716371802390022e-199,9.789903759004951e-184,2.1578926054830057e-178,2.247598456634206e-173,1.9895945650215745e-168,1.7203267870700736e-163,1.5310471774410184e-158,1.43565469886521e-153,1.4342482554455165e-148,1.534411871222153e-143,1.7613016412695512e-138,2.1692977044696332e-133,2.8636440447878334e-128,4.044212487843394e-123,6.096116823790158e-118,9.781977505545162e-113,1.666152845111448e-107,3.0035016717322656e-102,5.712882200575147e-97,1.1431151107694868e-91,2.399094223963611e-86,5.265981665989032e-81,1.2055493816808975e-75,2.8708897140266778e-70},

In [16]:
ii=-15
jj=-15
nt=1
nl=8
res1=Array{Float64,2}(undef,nl+1+4+2,nt)
for i in 1:nt
        ai1=i-1
        @everywhere t1=0.55-0.05*$ai1
#         println("t1=",t1)
        A=pi - asin(2*sqrt(2)/3);
        A1=A+ii/15
        A2=A-jj/15
        jp4=2*[1 1 1 1]
        # z1v=im*pv(jp4,A)[:,1]
        # z2v=im*pv(jp4,A)[:,2]
        # z3v=im*pv(jp4,A)[:,3]
        # z4v=im*pv(jp4,A)[:,4]
        z1v=im*pv2(jp4,A1,A2)[1][:,1]
        z2v=im*pv2(jp4,A1,A2)[1][:,2]
        z3v=im*pv2(jp4,A1,A2)[1][:,3]
        z4v=im*pv2(jp4,A1,A2)[1][:,4]
        @everywhere g4f1=slce2($z1v)
        @everywhere g4f2=slce2($z2v)
        @everywhere g4f3=slce2($z3v)
        @everywhere g4f4=slce2($z4v)
        nn=0
        jn=16
        js1=jn
        js2=jn
        js3=jn
        js4=jn
        k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
        g4tf1=transpose(conj(g4f1))
        g4tf2=transpose(conj(g4f2))
        g4tf3=transpose(conj(g4f3))
        g4tf4=transpose(conj(g4f4))
#         fr1=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#         fr2=(x)->itfro([x[1] x[2] x[3]],h4tf1,h4tf2,h4tf3,h4tf4,h4f1,h4f2,h4f3,h4f4,nn,t1,k)
#         (val,err) = hcubature(fr1, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         (val2,err2) = hcubature(fr2, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
#         val22=val2*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123test(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,jm,aj1,5/t1,100,at1,nl)
    res1[1,i]=abs(t1)
    res1[2,i]=A1
    res1[3,i]=A2
    res1[4,i]=abs(sum(at1[:,1]))
    res1[5,i]=abs(sum(at1[:,2]))
    for jj in 6:nl+1+4+2
        res1[jj,i]=abs(sum(at1[:,jj-3]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+1+4+2
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

h5open("./result.h5", "w") do file
    write(file, "result", res1)
end

{0.55,0.9106332362490182,2.9106332362490184,1.2422738830973867e12,7.251389078251448e11,7.866937357399338e11,4.922641211725253e12,1.6833966416548383e13,1.3974200535510439e14,7.871881166823578e14,8.106968458710493e15,3.12074528028144e12,3.12074528028144e12,3.1207452802814395e12,3.12074528028144e12},

# Test on Closed 4-Bridge

In [ ]:
l=11
nl=30
res1=Array{Float64,2}(undef,nl+1+4,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=5-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]*10
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*pv(jp4,A)[:,1]
    z2v=im*pv(jp4,A)[:,2]
    z3v=im*pv(jp4,A)[:,3]
    z4v=im*pv(jp4,A)[:,4]
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere g4f4=slce2($z4v)
    nn=0
    jn=19
    js1=jn
    js2=jn
    js3=jn
    js4=jn
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
    g4tf4=transpose(conj(g4f4))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123test_s(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,jm,aj1,5/t1,100,at1,nl+4)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+1+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+1+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result420a.jld", "key10",res1)

In [38]:
res1=load("~/julia code/data/result420a.jld", "key10")
for i in 1:size(res1)[2]
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

{2.51,4.5075207682785303e46,4.265692662831929e44,2.3862227007678227e48,1.3452468462406795e52,4.37348732681035e59,1.4646144248659934e67,5.049711066990503e74,1.791708457380734e82,6.539540914402828e89,2.454340163250025e97,9.468124493599561e104,3.7529620957856122e112,1.5279512971732162e120,6.387317358377903e127,2.7406511298419733e135,1.2066235064745008e143,5.449201220869646e150,2.5234868691629993e158,1.1979657244544466e166,5.828204062767297e173,2.9050105475722346e181,1.4830723446897928e189,7.75284039831451e196,4.1488564178152705e204,2.2722336387932146e212,1.273283832317012e220,7.298578783854999e227,4.2784623428621816e235,2.564313046131824e243,1.5710417384867591e251,9.836463243156656e258,6.29256730625138e266,4.112061253957844e274},{2.5,5.7817513733542645e47,5.4704901124014355e45,3.061354931879199e49,1.72645668958672e53,5.6161913535166504e60,1.8816744377498287e68,6.489981372736176e75,2.3032974880137875e83,8.407891366385132e90,3.155617863500128e98,1.2172432779232598e106,4.823991240893638e113,

In [82]:
l=25
nl=30
res1=Array{Float64,2}(undef,nl+1+4,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=10-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]*10
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*pv(jp4,A)[:,1]
    z2v=im*pv(jp4,A)[:,2]
    z3v=im*pv(jp4,A)[:,3]
    z4v=im*pv(jp4,A)[:,4]
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere g4f4=slce2($z4v)
    nn=0
    jn=10
    js1=jn
    js2=jn
    js3=jn
    js4=jn
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
    g4tf4=transpose(conj(g4f4))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123test_s(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,jm,aj1,5/t1,100,at1,nl+4)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+1+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+1+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result420a.jld", "key10",res1)

{10.0,1.1186754631339357e-159,1.1948781523442153e-161,5.21417101487884e-158,2.339283324986464e-154,4.7853765163050873e-147,1.0308742021504465e-139,2.39269271665513e-132,6.181754114508388e-125,1.846233106211907e-117,6.6340564137017e-110,2.96702446147897e-102,1.6557705737504594e-94,1.1028233548347854e-86,8.215267932121038e-79,6.4979983594259406e-71,5.292999580020978e-63,4.372574699668175e-55,3.638078867300927e-47,3.040463735956111e-39,2.551380461106008e-31,2.1523780336140512e-23,1.8308302275955478e-15,1.5785815105884965e-7,13.92078485672354,1.2741452402852027e9,1.2390800591890883e17,1.326002242751164e25,1.633801549713891e33,2.4135910078234545e41,4.315258618076167e49,9.048304769241024e57,2.1158296620747326e66,5.281018946305966e74,NaN},{9.8,2.291505141836144e-158,2.4953139046294115e-160,1.04836947081176e-156,4.5408516862416075e-153,8.826632347701972e-146,1.874305421128696e-138,4.483243456768971e-131,1.2476893377344193e-123,4.1762989618490125e-116,1.72836932003883e-108,8.821400773999858e-10

In [ ]:
l=11
nl=30
res1=Array{Float64,2}(undef,nl+1+4,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=5-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]*10
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*pv(jp4,A)[:,1]
    z2v=im*pv(jp4,A)[:,2]
    z3v=im*pv(jp4,A)[:,3]
    z4v=im*pv(jp4,A)[:,4]
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere g4f4=slce2($z4v)
    nn=0
    jn=16
    js1=jn
    js2=jn
    js3=jn
    js4=jn
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
    g4tf4=transpose(conj(g4f4))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123test_s(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,jm,aj1,5/t1,100,at1,nl+4)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+1+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+1+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result420a.jld", "key10",res1)

In [12]:
res1=load("~/julia code/data/result420.jld", "key10")
for i in 1:size(res1)[2]
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

{4.0,3.8616835854920054e143,3.762763345995896e141,1.9885874304976563e145,1.0650176650933775e149,3.1699897260769295e156,9.899443218314963e163,3.2401025060663803e171,1.110430690804862e179,3.9811611935770614e186,1.4918762201098478e194,5.838577167768547e201,2.3844920690930568e209,1.0154249159644961e217,4.504658148143803e224,2.0795822355302275e232,9.979129997708027e239,4.971985405690333e247,2.5696671888086716e255,1.3766584934263468e263,7.641365896243548e270,4.39313034017924e278,2.6153335583114353e286,1.6118277135898543e294},{3.9,1.0686156442284186e148,1.0393464212494541e146,5.512361024965537e149,2.9618061530617482e153,8.867171705656113e160,2.7828143663441824e168,9.14491278030532e175,3.1438082554919603e183,1.1295762915399708e191,4.2378372760900525e198,1.658407169605782e206,6.761991314944118e213,2.8695700613394894e221,1.2661589919259676e229,5.8041409488448255e236,2.762462602552144e244,1.3644348611007817e252,6.990733780799824e259,3.7138448343791363e267,2.044876179682312e275,1.1664384802014642e

In [ ]:
l=2
nl=20
res1=Array{Float64,2}(undef,nl+1+4,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=2.51-0.01*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]*10
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*pv(jp4,A)[:,1]
    z2v=im*pv(jp4,A)[:,2]
    z3v=im*pv(jp4,A)[:,3]
    z4v=im*pv(jp4,A)[:,4]
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere g4f4=slce2($z4v)
    nn=0
    jn=21
    js1=jn
    js2=jn
    js3=jn
    js4=jn
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
    g4tf4=transpose(conj(g4f4))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123test(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,jm,aj1,5/t1,100,at1,nl+4)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+1+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+1+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result420b.jld", "key21",res1)

In [19]:
res1=load("~/julia code/data/result420b.jld", "key21")
for i in 1:size(res1)[2]
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

{2.51,4.507520768278529e246,4.265692662831928e244,2.386222700767821e248,1.345246846240679e252,4.373487326810349e259,1.4646144248659929e267,5.049711066990501e274,1.7917084573807332e282,6.539540914402827e289,2.454340163250025e297,9.468124493599557e304,Inf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN},{2.5,5.781751373354263e247,5.470490112401433e245,3.0613549318791983e249,1.7264566895867194e253,5.616191353516649e260,1.881674437749828e268,6.489981372736175e275,2.303297488013787e283,8.40789136638513e290,3.155617863500127e298,1.2172432779232594e306,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN},

In [76]:
ii=-15
jj=-15
nt=25
nl=30
res1=Array{Float64,2}(undef,nl+1+4+2,nt)
for i in 1:nt
        ai1=i-1
        @everywhere t1=10-0.2*$ai1
#         println("t1=",t1)
        A=pi - asin(2*sqrt(2)/3);
        A1=A+ii/15
        A2=A-jj/15
        jp4=2*[1 1 1 1]*10
        # z1v=im*pv(jp4,A)[:,1]
        # z2v=im*pv(jp4,A)[:,2]
        # z3v=im*pv(jp4,A)[:,3]
        # z4v=im*pv(jp4,A)[:,4]
        z1v=im*pv2(jp4,A1,A2)[1][:,1]
        z2v=im*pv2(jp4,A1,A2)[1][:,2]
        z3v=im*pv2(jp4,A1,A2)[1][:,3]
        z4v=im*pv2(jp4,A1,A2)[1][:,4]
        @everywhere g4f1=slce2($z1v)
        @everywhere g4f2=slce2($z2v)
        @everywhere g4f3=slce2($z3v)
        @everywhere g4f4=slce2($z4v)
        nn=0
        jn=10
        js1=jn
        js2=jn
        js3=jn
        js4=jn
        k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
        g4tf1=transpose(conj(g4f1))
        g4tf2=transpose(conj(g4f2))
        g4tf3=transpose(conj(g4f3))
        g4tf4=transpose(conj(g4f4))
#         fr1=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#         fr2=(x)->itfro([x[1] x[2] x[3]],h4tf1,h4tf2,h4tf3,h4tf4,h4f1,h4f2,h4f3,h4f4,nn,t1,k)
#         (val,err) = hcubature(fr1, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         (val2,err2) = hcubature(fr2, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
#         val22=val2*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123test_s(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,jm,aj1,5/t1,100,at1,nl+4)
    res1[1,i]=abs(t1)
    res1[2,i]=A1
    res1[3,i]=A2
    res1[4,i]=abs(sum(at1[:,1]))
    res1[5,i]=abs(sum(at1[:,2]))
    for jj in 6:nl+1+4+2
        res1[jj,i]=abs(sum(at1[:,jj-3]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+1+4+2
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result4a20c1.jld", "key21",res1)

{10.0,0.9106332362490182,2.9106332362490184,1.701349736232087e-159,2.6222097025907796e-161,2.5090574502562992e-158,2.0780681688322256e-154,3.6773965831983686e-147,7.67475046221811e-140,1.7270403378925754e-132,4.2760554044630633e-125,1.209523907868464e-117,4.079440624131855e-110,1.707746351832276e-102,8.997727316102117e-95,5.760261055291607e-87,4.196255325560085e-79,3.282380821406992e-71,2.659648428029085e-63,2.1915186317734267e-55,1.820659762245072e-47,1.5196028525956665e-39,1.2730039076168999e-31,1.0710623790284743e-23,9.070277598813175e-16,7.762959969657188e-8,6.762625019420976,6.068270697578752e8,5.720443798642782e16,5.845174306921056e24,6.769070067465911e32,9.323254086695328e40,1.565272217661831e49,3.1383672928979602e57,7.1446563271685955e65,1.7575063096519776e74,NaN},{9.8,0.9106332362490182,2.9106332362490184,3.4609154849728586e-158,5.441185305296373e-160,5.004052265639314e-157,3.96420629319207e-153,6.584666276008993e-146,1.3342710869065541e-138,3.0458367307489646e-131,8.023256554

In [79]:
res1=load("~/julia code/data/result4a20c1.jld", "key21")
for i in 1:size(res1)[2]
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

{10.0,0.9106332362490182,2.9106332362490184,1.701349736232087e-159,2.6222097025907796e-161,2.5090574502562992e-158,2.0780681688322256e-154,3.6773965831983686e-147,7.67475046221811e-140,1.7270403378925754e-132,4.2760554044630633e-125,1.209523907868464e-117,4.079440624131855e-110,1.707746351832276e-102,8.997727316102117e-95,5.760261055291607e-87,4.196255325560085e-79,3.282380821406992e-71,2.659648428029085e-63,2.1915186317734267e-55,1.820659762245072e-47,1.5196028525956665e-39,1.2730039076168999e-31,1.0710623790284743e-23,9.070277598813175e-16,7.762959969657188e-8,6.762625019420976,6.068270697578752e8,5.720443798642782e16,5.845174306921056e24,6.769070067465911e32,9.323254086695328e40,1.565272217661831e49,3.1383672928979602e57},{9.8,0.9106332362490182,2.9106332362490184,3.4609154849728586e-158,5.441185305296373e-160,5.004052265639314e-157,3.96420629319207e-153,6.584666276008993e-146,1.3342710869065541e-138,3.0458367307489646e-131,8.02325655496376e-124,2.5262474641247367e-116,9.82475539194

In [ ]:
ii=-15
jj=-15
nt=11
nl=30
res1=Array{Float64,2}(undef,nl+1+4+2,nt)
for i in 1:nt
        ai1=i-1
        @everywhere t1=5-0.2*$ai1
#         println("t1=",t1)
        A=pi - asin(2*sqrt(2)/3);
        A1=A+ii/15
        A2=A-jj/15
        jp4=2*[1 1 1 1]*10
        # z1v=im*pv(jp4,A)[:,1]
        # z2v=im*pv(jp4,A)[:,2]
        # z3v=im*pv(jp4,A)[:,3]
        # z4v=im*pv(jp4,A)[:,4]
        z1v=im*pv2(jp4,A1,A2)[1][:,1]
        z2v=im*pv2(jp4,A1,A2)[1][:,2]
        z3v=im*pv2(jp4,A1,A2)[1][:,3]
        z4v=im*pv2(jp4,A1,A2)[1][:,4]
        @everywhere g4f1=slce2($z1v)
        @everywhere g4f2=slce2($z2v)
        @everywhere g4f3=slce2($z3v)
        @everywhere g4f4=slce2($z4v)
        nn=0
        jn=16
        js1=jn
        js2=jn
        js3=jn
        js4=jn
        k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
        g4tf1=transpose(conj(g4f1))
        g4tf2=transpose(conj(g4f2))
        g4tf3=transpose(conj(g4f3))
        g4tf4=transpose(conj(g4f4))
#         fr1=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#         fr2=(x)->itfro([x[1] x[2] x[3]],h4tf1,h4tf2,h4tf3,h4tf4,h4f1,h4f2,h4f3,h4f4,nn,t1,k)
#         (val,err) = hcubature(fr1, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         (val2,err2) = hcubature(fr2, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
#         val22=val2*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123test_s(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,jm,aj1,5/t1,100,at1,nl+4)
    res1[1,i]=abs(t1)
    res1[2,i]=A1
    res1[3,i]=A2
    res1[4,i]=abs(sum(at1[:,1]))
    res1[5,i]=abs(sum(at1[:,2]))
    for jj in 6:nl+1+4+2
        res1[jj,i]=abs(sum(at1[:,jj-3]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+1+4+2
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result4a20c2.jld", "key16",res1)

In [80]:
res1=load("~/julia code/data/result4a20c2.jld", "key16")
for i in 1:size(res1)[2]
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

{5.0,0.9106332362490182,2.9106332362490184,1.0080737288518264e-91,1.8749153754644186e-93,2.182138272026931e-90,9.803501916909565e-87,1.500071785402695e-79,3.4025282478333546e-72,9.405689752379409e-65,2.925537941063941e-57,9.888466767507527e-50,3.5769557512960863e-42,1.3753440869759215e-34,5.603308425149277e-27,2.414080788091447e-19,1.0974718416108979e-11,0.0005249013668809183,26314.618650586257,1.3774951705617097e12,7.505312472366167e19,4.2477428672375454e27,2.4955804102546814e35,1.522734242439702e43,9.661439496903818e50,6.38328248452859e58,4.3972014791247364e66,3.1610816685270475e74,2.372872861491145e82,1.860475094992792e90,1.5236088806669643e98,1.302525946284099e106,1.1608888409636552e114,1.0762768607561684e122},{4.8,0.9106332362490182,2.9106332362490184,5.626906828338463e-86,1.0324533824232576e-87,1.2067539766035456e-84,5.2797958729642255e-81,7.936978653740377e-74,1.7843129267194155e-66,4.929775046893678e-59,1.5416066893340703e-51,5.253929104203214e-44,1.9158243782650717e-36,7.39951

In [ ]:
ii=-15
jj=-15
nt=2
nl=30
res1=Array{Float64,2}(undef,nl+1+4+2,nt)
for i in 1:nt
        ai1=i-1
        @everywhere t1=2.51-0.01*$ai1
#         println("t1=",t1)
        A=pi - asin(2*sqrt(2)/3);
        A1=A+ii/15
        A2=A-jj/15
        jp4=2*[1 1 1 1]*10
        # z1v=im*pv(jp4,A)[:,1]
        # z2v=im*pv(jp4,A)[:,2]
        # z3v=im*pv(jp4,A)[:,3]
        # z4v=im*pv(jp4,A)[:,4]
        z1v=im*pv2(jp4,A1,A2)[1][:,1]
        z2v=im*pv2(jp4,A1,A2)[1][:,2]
        z3v=im*pv2(jp4,A1,A2)[1][:,3]
        z4v=im*pv2(jp4,A1,A2)[1][:,4]
        @everywhere g4f1=slce2($z1v)
        @everywhere g4f2=slce2($z2v)
        @everywhere g4f3=slce2($z3v)
        @everywhere g4f4=slce2($z4v)
        nn=0
        jn=21
        js1=jn
        js2=jn
        js3=jn
        js4=jn
        k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
        g4tf1=transpose(conj(g4f1))
        g4tf2=transpose(conj(g4f2))
        g4tf3=transpose(conj(g4f3))
        g4tf4=transpose(conj(g4f4))
#         fr1=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#         fr2=(x)->itfro([x[1] x[2] x[3]],h4tf1,h4tf2,h4tf3,h4tf4,h4f1,h4f2,h4f3,h4f4,nn,t1,k)
#         (val,err) = hcubature(fr1, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         (val2,err2) = hcubature(fr2, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
#         val22=val2*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123test_s(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,jm,aj1,5/t1,100,at1,nl+4)
    res1[1,i]=abs(t1)
    res1[2,i]=A1
    res1[3,i]=A2
    res1[4,i]=abs(sum(at1[:,1]))
    res1[5,i]=abs(sum(at1[:,2]))
    for jj in 6:nl+1+4+2
        res1[jj,i]=abs(sum(at1[:,jj-3]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+1+4+2
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result4a20a3.jld", "key21",res1)

In [69]:
res1=load("~/julia code/data/result4a20a3.jld", "key21")
for i in 1:size(res1)[2]
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

{2.51,1.577299902915685,2.2439665695823514,4.465337962993684e46,4.491013890415556e44,2.240209287352154e48,1.1537907531528071e52,3.2907219468017137e59,1.0109612521065115e67,3.289965508642061e74,1.1223951518831508e82,3.987177496689229e89,1.468229674595975e97,5.5872183395976124e104,2.1924935759347284e112,8.858388925489106e119,3.680891625150804e127,1.571659368883291e135,6.89088440107362e142,3.100704158515595e150,1.4312296737692698e158,6.773965332478098e165,3.2862540236918306e173,1.6335692815724883e181,8.317950403440498e188,4.337199096815727e196,2.3152330008871147e204,1.2648976202200306e212,7.070962747809896e219,4.043476868334645e227,2.3647127604668425e235,1.4139862859951575e243,8.642789325542498e250,5.398898234470999e258},{2.5,1.577299902915685,2.2439665695823514,5.72768798413938e47,5.759394554277312e45,2.8740027356984814e49,1.4805860668624788e53,4.223965333242763e60,1.2978231601775983e68,4.223607597557504e75,1.4408546268802622e83,5.117953496350326e90,1.8843211774635926e98,7.16900163173371

In [ ]:
ii=-15
jj=-15
nt=2
nl=30
res1=Array{Float64,2}(undef,nl+1+4+2,nt)
for i in 1:nt
        ai1=i-1
        @everywhere t1=2.51-0.01*$ai1
#         println("t1=",t1)
        A=pi - asin(2*sqrt(2)/3);
        A1=A+ii/15
        A2=A-jj/15
        jp4=2*[1 1 1 1]*10
        # z1v=im*pv(jp4,A)[:,1]
        # z2v=im*pv(jp4,A)[:,2]
        # z3v=im*pv(jp4,A)[:,3]
        # z4v=im*pv(jp4,A)[:,4]
        z1v=im*pv2(jp4,A1,A2)[1][:,1]
        z2v=im*pv2(jp4,A1,A2)[1][:,2]
        z3v=im*pv2(jp4,A1,A2)[1][:,3]
        z4v=im*pv2(jp4,A1,A2)[1][:,4]
        @everywhere g4f1=slce2($z1v)
        @everywhere g4f2=slce2($z2v)
        @everywhere g4f3=slce2($z3v)
        @everywhere g4f4=slce2($z4v)
        nn=0
        jn=21
        js1=jn
        js2=jn
        js3=jn
        js4=jn
        k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
        g4tf1=transpose(conj(g4f1))
        g4tf2=transpose(conj(g4f2))
        g4tf3=transpose(conj(g4f3))
        g4tf4=transpose(conj(g4f4))
#         fr1=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#         fr2=(x)->itfro([x[1] x[2] x[3]],h4tf1,h4tf2,h4tf3,h4tf4,h4f1,h4f2,h4f3,h4f4,nn,t1,k)
#         (val,err) = hcubature(fr1, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         (val2,err2) = hcubature(fr2, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
#         val22=val2*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123test_s(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,jm,aj1,5/t1,100,at1,nl+4)
    res1[1,i]=abs(t1)
    res1[2,i]=A1
    res1[3,i]=A2
    res1[4,i]=abs(sum(at1[:,1]))
    res1[5,i]=abs(sum(at1[:,2]))
    for jj in 6:nl+1+4+2
        res1[jj,i]=abs(sum(at1[:,jj-3]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+1+4+2
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result4a20c3.jld", "key21",res1)

In [81]:
res1=load("~/julia code/data/result4a20c3.jld", "key21")
for i in 1:size(res1)[2]
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

{2.51,0.9106332362490182,2.9106332362490184,3.807051654031222e46,7.449306894552742e44,9.494300559938993e47,2.794735237002108e51,2.8874678592319326e58,4.643702271035726e65,9.760842187845428e72,2.45808178414786e80,7.04478657144302e87,2.2232657308226017e95,7.557827859598599e102,2.7261062223764037e110,1.0325371555246874e118,4.076976766746439e125,1.669699726262555e133,7.067327490619885e140,3.083798233138955e148,1.3846365949493802e156,6.388835702533344e163,3.026261430813563e171,1.4704606107811094e179,7.324800058433736e186,3.7386546048830116e194,1.9544649166746446e202,1.0460999514252031e210,5.730707864617316e217,3.212200799341064e225,1.841769249597745e233,1.0799198846057896e241,6.473847816306332e248,3.9668234032016865e256},{2.5,0.9106332362490182,2.9106332362490184,4.881561860569456e47,9.55169903026753e45,1.2178499386979853e49,3.5772000259043147e52,3.6865340029676323e59,5.9137273513710244e66,1.2401364548306498e74,3.116603004688746e81,8.916123954474238e88,2.809521435981451e96,9.538129713208073

In [42]:
res1

27×11 Matrix{Float64}:
 5.0          4.8          4.6          …    3.2            3.0
 1.84397      1.84397      1.84397           1.84397        1.84397
 1.9773       1.9773       1.9773            1.9773         1.9773
 1.00473e109  5.63205e114  1.02833e121       8.91577e186    2.64141e201
 1.00077e107  5.59197e112  1.01596e119       8.57289e184    2.52986e199
 5.07168e110  2.85134e116  5.23026e122  …    4.64998e188    1.38273e203
 2.61934e114  1.48141e120  2.74117e126       2.55102e192    7.63768e206
 7.32937e121  4.19696e127  7.8862e133        7.92445e199    2.4002e214
 2.17979e129  1.26424e135  2.40331e141       2.56227e207    7.83133e221
 6.87861e136  4.04035e142  7.73068e148       8.60956e214    2.64906e229
 2.30076e144  1.36779e150  2.61811e156  …    3.00284e222    9.28017e236
 8.14887e151  4.89623e157  9.31995e163       1.0861e230     3.36406e244
 3.05344e159  1.84885e165  3.48416e171       4.07039e237    1.26096e252
 ⋮                                      ⋱                  

In [ ]:
ii=-15
jj=-15
nt=11
nl=20
res1=Array{Float64,2}(undef,nl+1+4+2,nt)
for i in 1:nt
        ai1=i-1
        @everywhere t1=5-0.2*$ai1
#         println("t1=",t1)
        A=pi - asin(2*sqrt(2)/3);
        A1=A+ii/15
        A2=A-jj/15
        jp4=2*[1 1 1 1]*10
        # z1v=im*pv(jp4,A)[:,1]
        # z2v=im*pv(jp4,A)[:,2]
        # z3v=im*pv(jp4,A)[:,3]
        # z4v=im*pv(jp4,A)[:,4]
        z1v=im*pv2(jp4,A1,A2)[1][:,1]
        z2v=im*pv2(jp4,A1,A2)[1][:,2]
        z3v=im*pv2(jp4,A1,A2)[1][:,3]
        z4v=im*pv2(jp4,A1,A2)[1][:,4]
        @everywhere g4f1=slce2($z1v)
        @everywhere g4f2=slce2($z2v)
        @everywhere g4f3=slce2($z3v)
        @everywhere g4f4=slce2($z4v)
        nn=0
        jn=16
        js1=jn
        js2=jn
        js3=jn
        js4=jn
        k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
        g4tf1=transpose(conj(g4f1))
        g4tf2=transpose(conj(g4f2))
        g4tf3=transpose(conj(g4f3))
        g4tf4=transpose(conj(g4f4))
#         fr1=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#         fr2=(x)->itfro([x[1] x[2] x[3]],h4tf1,h4tf2,h4tf3,h4tf4,h4f1,h4f2,h4f3,h4f4,nn,t1,k)
#         (val,err) = hcubature(fr1, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         (val2,err2) = hcubature(fr2, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#         val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
#         val22=val2*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123test_s(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,jm,aj1,5/t1,100,at1,nl+4)
    res1[1,i]=abs(t1)
    res1[2,i]=A1
    res1[3,i]=A2
    res1[4,i]=abs(sum(at1[:,1]))
    res1[5,i]=abs(sum(at1[:,2]))
    for jj in 6:nl+1+4+2
        res1[jj,i]=abs(sum(at1[:,jj-3]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+1+4+2
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

save("~/julia code/data/result4a20c1.jld", "key21",res1)

In [54]:
nl=20
res1=load("~/julia code/data/result4a20c1.jld", "key21")
for i in 1:size(res1)[2]
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

{5.0,0.9106332362490182,2.9106332362490184,1.0080737288518264e-91,1.8749153754644186e-93,2.1821382720269312e-90,9.803501916909567e-87,1.5000717854026957e-79,3.4025282478333546e-72,9.405689752379409e-65,2.925537941063941e-57,9.888466767507524e-50,3.5769557512960863e-42,1.3753440869759215e-34,5.603308425149275e-27,2.4140807880914473e-19,1.0974718416108978e-11,0.0005249013668809184,26314.618650586257,1.3774951705617095e12,7.505312472366168e19,4.247742867237545e27,2.4955804102546814e35,1.5227342424397018e43},{4.8,0.9106332362490182,2.9106332362490184,5.6269068283384626e-86,1.0324533824232576e-87,1.2067539766035456e-84,5.279795872964226e-81,7.936978653740377e-74,1.784312926719416e-66,4.929775046893679e-59,1.5416066893340706e-51,5.253929104203214e-44,1.9158243782650717e-36,7.399515766429392e-29,3.0078552856480695e-21,1.281364129932482e-13,5.7060611375513924e-6,265.3105168972724,1.2879096897793465e10,6.530183065108995e17,3.4602905055733335e25,1.9169636116153768e33,1.1104605572414109e41,6.7264

In [46]:
nl

30

# Open 3-Vertex

In [ ]:
l=1
nl=20
res1=Array{Float64,2}(undef,nl+4+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=0.625-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[2 0 0]*2.5
    z2v=im*[0 2 0]*2.5
    z3v=im*[0 0 2]*2.5
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    nn=0
    jn=16
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    @time tran123gv3(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

h5open("./result.h5", "w") do file
    write(file, "result", res1)
end

In [9]:
l=25
nl=20
res1=Array{Float64,2}(undef,nl+4+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=10-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[2 0 0]*2.5
    z2v=im*[0 2 0]*2.5
    z3v=im*[0 0 2]*2.5
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    nn=0
    jn=8
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

h5open("./result.h5", "w") do file
    write(file, "result", res1)
end

{10.0,236278.2473047212,22827.03090831267,3.501983509754209e6,3.201382212019725e9,1.603710795942244e15,1.4992899473612913e21,2.452639953091408e27,5.717001116514918e33,1.6205720021561103e40,5.273270151529238e46,2.0170641160126864e53,9.905305256278027e59,6.96997598677234e66,7.324665356871207e73,1.0823799848115057e81,2.039878012421022e88,4.503373043457339e95,1.095002970792534e103,2.8188921303541727e110,7.515895524547144e117,2.0555828658201028e125,5.760843671868481e132,1.663953680067983e140},{9.8,405331.46528649784,38574.16362742613,6.09219714100335e6,5.471221312719523e9,2.849434387655021e15,2.7926200680654307e21,4.5709415554094077e27,1.0147462822315151e34,2.686829937014801e40,8.251682029229437e46,3.093369152136597e53,1.5725254812501726e60,1.1857411278031938e67,1.3143572545734155e74,1.9609128719664137e81,3.5752900692018776e88,7.391798029824653e95,1.6476718162699906e103,3.843686082489982e110,9.248492222014744e117,2.286357427571767e125,5.831131118465095e132,1.5519583476060655e140},{9.6,71520

In [11]:
l=10
nl=20
res1=Array{Float64,2}(undef,nl+4+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=2-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[2 0 0]*5
    z2v=im*[0 2 0]*5
    z3v=im*[0 0 2]*5
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    nn=0
    jn=12
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

h5open("./result.h5", "w") do file
    write(file, "result", res1)
end

{2.0,2.4960201969670852e58,9.200306813944181e56,6.931388916550661e59,5.616388390921446e62,4.276264764746622e68,3.813886513445139e74,3.886872520217931e80,4.460769258019533e86,5.705227551018744e92,8.064200351951139e98,1.2506187169007107e105,2.1138999077483666e111,3.8700827050078566e117,7.628076305083635e123,1.6092656288391932e130,3.61316574515533e136,8.586219965068922e142,2.148160383974411e149,5.629683207942443e155,1.5380690117142135e162,4.361104100191057e168,1.278028603418755e175,3.8560981134485384e181},{1.8,5.841625102823458e65,2.1265788577834604e64,1.6342594730406636e67,1.3323808389595668e70,1.0032539921367717e76,8.613581337999403e81,8.232266507737502e87,8.622473435190148e93,9.780350460362096e99,1.1896261529772028e106,1.5385245857486722e112,2.0997946036202115e118,3.004207342749111e124,4.479027188421144e130,6.922215733785181e136,1.1037806045083668e143,1.8084384349268809e149,3.0334149167059185e155,5.192617463293574e161,9.046111226743437e167,1.5999616521381856e174,2.8669413900816663e180,

LoadError: InterruptException:

In [ ]:
l=1
nl=20
res1=Array{Float64,2}(undef,nl+4+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=2.4-0.05*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[2 0 0]*10
    z2v=im*[0 2 0]*10
    z3v=im*[0 0 2]*10
    w1v=im*[2 0 0]*5
    w2v=im*[0 2 0]*5
    w3v=im*[0 0 2]*5
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=14
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

h5open("./result.h5", "w") do file
    write(file, "result", res1)
end

In [9]:
l=1
nl=25
res1=Array{Float64,2}(undef,nl+4+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=4.2-0.01*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[2 0 0]*10
    z2v=im*[0 2 0]*10
    z3v=im*[0 0 2]*10
    w1v=im*[2 0 0]*5
    w2v=im*[0 2 0]*5
    w3v=im*[0 0 2]*5
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=10
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=sqrt(abs(sum(at1[:,2]))*abs(sum(at1[:,3])))
    res1[4,i]=abs(sum(at1[:,2]))
    res1[5,i]=abs(sum(at1[:,3]))
    for jj in 5:nl+4
        res1[jj,i]=abs(sum(at1[:,jj-2]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

h5open("./result.h5", "w") do file
    write(file, "result", res1)
end

{4.2,3.637797043880759e56,3.761861909574107e62,1.7516572187136734e58,4.365537652931724e61,3.1961888674126247e68,2.7699711938060934e75,2.7529050494203404e82,3.0874476872702633e89,3.8663565483812786e96,5.363907829199485e103,8.193940601471156e110,1.371685661998333e118,2.5065887704207667e125,4.983526763887938e132,1.0746837182984647e140,2.5060845205443082e147,6.300130912636265e154,1.7022526628740827e162,4.929203734406819e169,1.5258314434718896e177,5.0384177830531005e184,1.7717568136672432e192,6.625926614983551e199,2.632160082875006e207,1.109426366574865e215,4.9550440755644665e222,2.3415951899267103e230,1.1688309121403291e238},

In [7]:
3.047299593407282/3.047354759508687

0.9999818970530974

In [19]:
l=1
nl=20
res1=Array{Float64,2}(undef,nl+4+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=4-0.2*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[2 0 0]*10
    z2v=im*[0 2 0]*10
    z3v=im*[0 0 2]*10
    w1v=im*[2 0 0]*5
    w2v=im*[0 2 0]*5
    w3v=im*[0 0 2]*5
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    nn=0
    jn=12
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    tran123gv3p(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,h4f1,h4f2,h4f3,jm,aj1,5/t1,100,at1,nl+4,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+4
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+4
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

h5open("./result.h5", "w") do file
    write(file, "result", res1)
end

{5.0,1.9506759756903235e45,4.476010620162131e43,9.092360963271929e46,2.180297707487308e50,1.5032617843171468e57,1.26186335530079e64,1.2431777328534992e71,1.4114629098477254e78,1.8228178684998466e85,2.6482800257438976e92,4.2889153579168293e99,7.688732670175171e106,1.5184193127798974e114,3.29309795104686e121,7.825857684611001e128,2.0334352880288897e136,5.760981297040831e143,1.773216169144046e151,5.904320262520403e158,2.117302365333658e166,8.143774739364592e173,3.348807401592234e181,1.4690655781498845e189},{4.8,5.6551659060127875e47,1.2798835188302194e46,2.65694272627601e49,6.429232461005278e52,4.506590984276732e59,3.823612130373598e66,3.7833164542428514e73,4.282339055014859e80,5.474020959832169e87,7.829558772454421e94,1.2449672569811337e102,2.1907357537349473e109,4.2508282952664215e116,9.063344889079897e123,2.1150689647734272e131,5.378324203440312e138,1.4832960845591595e146,4.417311839563638e153,1.415292990267744e161,4.865589766735705e168,1.791866221831943e176,7.06271641182448e183,2.9779

In [14]:
l=1
nl=20
res1=Array{Float64,2}(undef,nl+1,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=5-0.05*$ai1
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[2 0 0]*10
    z2v=im*[0 2 0]*10
    z3v=im*[0 0 2]*10
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    nn=0
    jn=12
    js1=jn
    js2=jn
    js3=jn
    js4=js1+js3+js3
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
#     f=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-10, abstol=1e-10, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),nl+4)
    @time tran123gv3(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,jm,aj1,5/t1,100,at1,nl,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nl+1
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nl+1
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
end

h5open("./result.h5", "w") do file
    write(file, "result", res1)
end

355.752813 seconds (13.53 k allocations: 1.105 MiB)
{5.0,9.425588928883827e84,1.294823847406456e83,7.087441366766297e86,4.230367037907098e90,2.6431374098829596e94,1.7155981826832646e98,1.1484252843456034e102,7.906589521616484e105,5.582680895860171e109,4.037499830970931e113,2.9863818435466767e117,2.2574398349805967e121,1.7421728722537295e125,1.3719301588904728e129,1.1015407251819188e133,9.01349078481886e136,7.511411203532548e140,6.372375832402894e144,5.500234436140963e148,4.828323120031502e152},

In [15]:
l=1
res1=Array{Float64,2}(undef,l,3)
ij1=[0 0]
for i in 1:l
    ai1=i-1
    t1=0.8+ai1*(0.1)
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=2*[1.0 0.0 0.0]*im
    z2v=2*[0.0 1.0 0.0]*im
    z3v=2*[0.0 0.0 1.0]*im
    z4v=im*pv(jp4,A)[:,4]
    g01=slce2(z1v)
    g02=slce2(z2v)
    g03=slce2(z3v)
#     println(g01)
#     println(g02)
#     println(g03)
    jcap=10
    jtag=0
    res1[i,1]=0
    res1[i,2]=0
    res1[i,3]=0
    for jj in 0:(2*jcap+1)^3-1
        j1=rem(jj,(2*jcap+1))
        j2=rem(fld(jj,(2*jcap+1)),(2*jcap+1))
        j3=rem(fld(fld(jj,(2*jcap+1)),(2*jcap+1)),(2*jcap+1))
        if j3!=jtag
            println(j3)
        end
        if Int(2*(j1/2+j2/2+j3/2))%2==0
            TM3r=TransformM3([j1 j2 j3]/2,t1,g01,g02,g03,jm,aj1,ij1)
            state=0.0*im
#             print(size(TM3r,1))
            for mi in 1:size(TM3r,1)
                m3=j3/2-rem(mi-1,j3+1)
                m2=j2/2-rem(fld(mi-1,j3+1),j2+1)
                m1=j1/2-rem(fld(fld(mi-1,j3+1),j2+1),j1+1)
#                 println([j1/2 j2/2 j3/2 m1 m2 m3])
                if m3 == -m1-m2
                    state=state+TM3r[mi]*Wj1([j1/2 j2/2 j3/2 m1 m2 -m1-m2],jm,aj1,ij1)
                end
            end
            res1[i,1]=res1[i,1]+adjoint(state)*state
            res1[i,2]=res1[i,2]+adjoint(state)*state*(j1/2*(j1/2+1))*t1^2
        end
        jtag=j3
    end
    res1[i,3]=abs(res1[i,2]/res1[i,1])
end
res1

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20


1×3 Matrix{Float64}:
 5651.49  17280.7  3.05772

In [12]:
l=1
res1=zeros(l,3)
ij1=[0 0]
for i in 1:l
    ai1=i-1
    t1=0.8+ai1*(0.1)
    A=pi - asin(2*sqrt(2)/3);
    jp4=2*[1 1 1 1]
    z1v=im*pv(jp4,A)[:,1]
    z2v=im*pv(jp4,A)[:,2]
    z3v=im*pv(jp4,A)[:,3]
    z4v=im*pv(jp4,A)[:,4]
#     z1v=2*[1.0 0.0 0.0]*im
#     z2v=2*[0.0 1.0 0.0]*im
#     z3v=2*[0.0 0.0 1.0]*im
#     z4v=im*pv(jp4,A)[:,4]
    g01=slce2(z1v)
    g02=slce2(z2v)
    g03=slce2(z3v)
    g04=slce2(z4v)
#     println(g01)
#     println(g02)
#     println(g03)
    jcap=4
    jtag=0
    res1[i,1]=0
    res1[i,2]=0
    res1[i,3]=0
    jlist111=zeros((2*jcap+1)^4)
    jlist111.=[jj for jj in 0:(2*jcap+1)^4-1]
    randl(jlist111)
    jlist111=Int.(jlist111)
        jj=200
        j1=rem(jj,(2*jcap+1))
        j2=rem(fld(jj,(2*jcap+1)),(2*jcap+1))
        j3=rem(fld(fld(jj,(2*jcap+1)),(2*jcap+1)),(2*jcap+1))
        j4=rem(fld(fld(fld(jj,(2*jcap+1)),(2*jcap+1)),(2*jcap+1)),(2*jcap+1))
#         if j3!=jtag
#             println(jjl)
#         end
#         println(j1/2,j2/2,j3/2,j4/2)
        ablist=Fai4([j1 j2 j3 j4]/2)
        if Int(2*(j1/2+j2/2+j3/2+j4/2))%2==0 && size(ablist,1)!=0
            TM3r=TransformM34([j1 j2 j3 j4]/2,t1,g01,g02,g03,g04,jm,aj1,ij1)
            TM4r=Trans4ott([j1 j2 j3 j4]/2,t1,g01,g02,g03,g04,jm,aj1,ablist,size(ablist,1))
#             println(TM3r,size(TM3r))
#             println(TM4r,size(TM4r,1),size(ablist))
            state=zeros(size(ablist,1),size(ablist,1))*im
#             print(size(TM3r,1))
            for mi in 1:size(TM3r,2)
                m4=j4/2-rem(mi-1,j4+1)
                m3=j3/2-rem(fld(mi-1,j4+1),j3+1)
                m2=j2/2-rem(fld(fld(mi-1,j4+1),j3+1),j2+1)
                m1=j1/2-rem(fld(fld(fld(mi-1,j4+1),j3+1),j2+1),j1+1)
#                 println([j1/2 j2/2 j3/2 m1 m2 m3])
                if m4 == -m1-m2-m3
                    coef1=(-1)^(Int(j4/2+j3/2+j2/2+j1/2-m1-m2-m3-(-m1-m2-m3)))
                    for ii in 1:size(ablist,1)
                        state[:,ii].=state[:,ii].+(TM3r[:,mi]*Wj1([j1/2 j2/2 ablist[ii,2] m1 m2 -m1-m2],jm,aj1,ij1))*Wj1([ablist[ii,2] j3/2 j4/2 m1+m2 m3 -m1-m2-m3],jm,aj1,ij1)*(-1)^(ablist[i,2]+m1+m2)
                    end
                end
            end
            for ii in 1:size(ablist,1)
                res1[i,1]=res1[i,1]+abs(adjoint(state[:,ii])*state[:,ii])
                res1[i,2]=res1[i,2]+abs(adjoint(state[:,ii])*state[:,ii]*(j1/2*(j1/2+1))*t1^2)
            end
        end
    res1[i,3]=abs(res1[i,2]/res1[i,1])
end
res1

1.02.01.00.0
f1=[1.0 2.0 1.0 0.0 -1.0 0.0 1.0 -0.0]0.0 + 0.0im
g=2.9134236986363695 - 0.0im2.2835178083939853 - 0.0im0.0 - 0.0im1.0 - 0.0im
f1=[1.0 2.0 1.0 0.0 0.0 -1.0 1.0 -0.0]0.0 + 0.0im
g=1.1797372979173577 + 2.0433649395768843im-3.2558329500411274 - 0.0im0.0 - 0.0im1.0 - 0.0im
f1=[1.0 2.0 1.0 0.0 1.0 -2.0 1.0 -0.0]0.0 + 0.0im
g=-0.47771290277784606 + 0.827423019042441im2.8427289942586533 - 0.0im0.0 - 0.0im1.0 - 0.0im
f1=[1.0 2.0 1.0 0.0 -1.0 1.0 0.0 -0.0]0.0 + 0.0im
g=2.9134236986363695 - 0.0im-1.0677152178393343 - 0.0im0.0 - 0.0im1.0 - 0.0im
f1=[1.0 2.0 1.0 0.0 0.0 0.0 0.0 -0.0]0.0 + 0.0im
g=1.1797372979173577 + 2.0433649395768843im2.2835178083939853 - 0.0im0.0 - 0.0im1.0 - 0.0im
f1=[1.0 2.0 1.0 0.0 1.0 -1.0 0.0 -0.0]0.0 + 0.0im
g=-0.47771290277784606 + 0.827423019042441im-3.2558329500411274 - 0.0im0.0 - 0.0im1.0 - 0.0im
f1=[1.0 2.0 1.0 0.0 -1.0 2.0 -1.0 -0.0]0.04195433297631042 + 0.0im
g=2.9134236986363695 - 0.0im0.30571876483148713 - 0.0im0.1053261180783228 - 0.0im1.0 - 0.0im
f

1×3 Matrix{Float64}:
 139.63  178.726  1.28

In [10]:
l1=0
l2=0
res1=Array{ComplexF64,3}(undef,7,2*l1+1,2*l2+1)
for ii in -l1:l1
    for jj in -l2:l2
    @everywhere t1=2.5
    jp6=2*[1 1 1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    xy=ii/30
    z=1-abs(jj/30)
    z1v=im*[0.0 0.0 jp6[1]]
    z4v=im*[0.0 0.0 -jp6[1]]
    z2v=im*[-sqrt(1-z^2) 0.0 -z]*jp6[1]
    z5v=im*[sqrt(1-z^2) 0.0 z]*jp6[1]
    z3v=im*[xy*sqrt(1-z^2) sqrt(1-xy^2)*sqrt(1-z^2) -z]*jp6[1]
    z6v=im*[-xy*sqrt(1-z^2) -sqrt(1-xy^2)*sqrt(1-z^2) z]*jp6[1]
    w1v=im*[0.0 0.0 jp6[1]]
    w4v=im*[0.0 0.0 -jp6[1]]
    w2v=im*[-sqrt(1-z^2) 0.0 -z]*jp6[1]
    w5v=im*[sqrt(1-z^2) 0.0 z]*jp6[1]
    w3v=im*[xy*sqrt(1-z^2) sqrt(1-xy^2)*sqrt(1-z^2) -z]*jp6[1]
    w6v=im*[-xy*sqrt(1-z^2) -sqrt(1-xy^2)*sqrt(1-z^2) z]*jp6[1]
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere g4f4=slce2($z4v)
    @everywhere g4f5=slce2($z5v)
    @everywhere g4f6=slce2($z6v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    @everywhere h4f4=slce2($w4v)
    @everywhere h4f5=slce2($w5v)
    @everywhere h4f6=slce2($w6v)
    nn=0
    jn=4
    js1=jn
    js2=jn
    js3=jn
    js4=jn
    js5=jn
    js6=jn
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
    g4tf4=transpose(conj(g4f4))
    g4tf5=transpose(conj(g4f5))
    g4tf6=transpose(conj(g4f6))
    h4tf1=transpose(conj(h4f1))
    h4tf2=transpose(conj(h4f2))
    h4tf3=transpose(conj(h4f3))
    h4tf4=transpose(conj(h4f4))
    h4tf5=transpose(conj(h4f5))
    h4tf6=transpose(conj(h4f6))
#     f=(x)->itfro6([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4tf5,g4tf6,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^6
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1)*(js5+1)*(js6+1),4)
    @time p6(js1,js2,js3,js4,js5,js6,t1,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,h4f1,h4f2,h4f3,h4f4,h4f5,h4f6,jm,aj1,5/t1,100,at1,jm61,jm62,B1,B2)
    res1[1,l1+1+ii,l2+1+jj]=t1
    res1[2,l1+1+ii,l2+1+jj]=sum(at1[:,1])/sqrt(sum(at1[:,3])*sum(at1[:,4]))
    res1[3,l1+1+ii,l2+1+jj]=sum(at1[:,2])/sqrt(sum(at1[:,3])*sum(at1[:,4]))
    res1[4,l1+1+ii,l2+1+jj]=sum(at1[:,3])
    res1[5,l1+1+ii,l2+1+jj]=sum(at1[:,4])
    res1[6,l1+1+ii,l2+1+jj]=xy
    res1[7,l1+1+ii,l2+1+jj]=z
#     print("{",res1[1,i],",",res1[2,i],",",res1[3,i],"},")
    end
end
print("{")
for j in 1:size(res1,2)
    for k in 1:size(res1,3)
        print("{",abs(res1[1,j,k]),",",abs(res1[2,j,k]),",",abs(res1[3,j,k]),",",abs(res1[4,j,k]),",",abs(res1[5,j,k]),",",res1[6,j,k],",",res1[7,j,k],"},")
    end
end
print("}")

 84.921693 seconds (1.64 M allocations: 98.630 MiB, 0.42% compilation time)
{{2.5,0.7761714702270126,0.9999999999999999,470.69931007759817,470.69931007759817,0.0 + 0.0im,1.0 + 0.0im},}

In [11]:
l1=0
l2=0
res1=Array{ComplexF64,3}(undef,7,2*l1+1,2*l2+1)
for ii in -l1:l1
    for jj in -l2:l2
    @everywhere t1=2.5
    jp6=2*[1 1 1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    xy=ii/30
    z=1-abs(jj/30)
    z1v=im*[0.0 0.0 jp6[1]]
    z4v=im*[0.0 0.0 -jp6[1]]
    z2v=im*[-sqrt(1-z^2) 0.0 -z]*jp6[1]
    z5v=im*[sqrt(1-z^2) 0.0 z]*jp6[1]
    z3v=im*[xy*sqrt(1-z^2) sqrt(1-xy^2)*sqrt(1-z^2) -z]*jp6[1]
    z6v=im*[-xy*sqrt(1-z^2) -sqrt(1-xy^2)*sqrt(1-z^2) z]*jp6[1]
    w1v=im*[0.0 0.0 jp6[1]]
    w4v=im*[0.0 0.0 -jp6[1]]
    w2v=im*[-sqrt(1-z^2) 0.0 -z]*jp6[1]
    w5v=im*[sqrt(1-z^2) 0.0 z]*jp6[1]
    w3v=im*[xy*sqrt(1-z^2) sqrt(1-xy^2)*sqrt(1-z^2) -z]*jp6[1]
    w6v=im*[-xy*sqrt(1-z^2) -sqrt(1-xy^2)*sqrt(1-z^2) z]*jp6[1]
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere g4f4=slce2($z4v)
    @everywhere g4f5=slce2($z5v)
    @everywhere g4f6=slce2($z6v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    @everywhere h4f4=slce2($w4v)
    @everywhere h4f5=slce2($w5v)
    @everywhere h4f6=slce2($w6v)
    nn=0
    jn=5
    js1=jn
    js2=jn
    js3=jn
    js4=jn
    js5=jn
    js6=jn
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
    g4tf4=transpose(conj(g4f4))
    g4tf5=transpose(conj(g4f5))
    g4tf6=transpose(conj(g4f6))
    h4tf1=transpose(conj(h4f1))
    h4tf2=transpose(conj(h4f2))
    h4tf3=transpose(conj(h4f3))
    h4tf4=transpose(conj(h4f4))
    h4tf5=transpose(conj(h4f5))
    h4tf6=transpose(conj(h4f6))
#     f=(x)->itfro6([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4tf5,g4tf6,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^6
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1)*(js5+1)*(js6+1),4)
    @time p6(js1,js2,js3,js4,js5,js6,t1,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,h4f1,h4f2,h4f3,h4f4,h4f5,h4f6,jm,aj1,5/t1,100,at1,jm61,jm62,B1,B2)
    res1[1,l1+1+ii,l2+1+jj]=t1
    res1[2,l1+1+ii,l2+1+jj]=sum(at1[:,1])/sqrt(sum(at1[:,3])*sum(at1[:,4]))
    res1[3,l1+1+ii,l2+1+jj]=sum(at1[:,2])/sqrt(sum(at1[:,3])*sum(at1[:,4]))
    res1[4,l1+1+ii,l2+1+jj]=sum(at1[:,3])
    res1[5,l1+1+ii,l2+1+jj]=sum(at1[:,4])
    res1[6,l1+1+ii,l2+1+jj]=xy
    res1[7,l1+1+ii,l2+1+jj]=z
#     print("{",res1[1,i],",",res1[2,i],",",res1[3,i],"},")
    end
end
print("{")
for j in 1:size(res1,2)
    for k in 1:size(res1,3)
        print("{",abs(res1[1,j,k]),",",abs(res1[2,j,k]),",",abs(res1[3,j,k]),",",abs(res1[4,j,k]),",",abs(res1[5,j,k]),",",abs(res1[6,j,k]),",",abs(res1[7,j,k]),"},")
    end
end
print("}")

4676.246695 seconds (191.76 k allocations: 13.307 MiB)
{{2.5,0.7761716894081577,1.0,470.70077556143815,470.70077556143815,0.0,1.0},}

In [ ]:
l1=12
l2=12
res1=Array{ComplexF64,3}(undef,7,2*l1+1,2*l2+1)
for ii in -l1:l1
    for jj in -l2:l2
    @everywhere t1=2.5
    jp6=2.0*[1 1 1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    xy=ii/30
    z=1-abs(jj/30)
    z1v=im*[0.0 0.0 jp6[1]]
    z4v=im*[0.0 0.0 -jp6[1]]
    z2v=im*[-sqrt(1-z^2) 0.0 -z]*jp6[1]
    z5v=im*[sqrt(1-z^2) 0.0 z]*jp6[1]
    z3v=im*[xy*sqrt(1-z^2) sqrt(1-xy^2)*sqrt(1-z^2) -z]*jp6[1]
    z6v=im*[-xy*sqrt(1-z^2) -sqrt(1-xy^2)*sqrt(1-z^2) z]*jp6[1]
    w1v=im*[0.0 0.0 jp6[1]]
    w4v=im*[0.0 0.0 -jp6[1]]
    w2v=im*[-sqrt(1-z^2) 0.0 -z]*jp6[1]
    w5v=im*[sqrt(1-z^2) 0.0 z]*jp6[1]
    w3v=im*[xy*sqrt(1-z^2) sqrt(1-xy^2)*sqrt(1-z^2) -z]*jp6[1]
    w6v=im*[-xy*sqrt(1-z^2) -sqrt(1-xy^2)*sqrt(1-z^2) z]*jp6[1]
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere g4f4=slce2($z4v)
    @everywhere g4f5=slce2($z5v)
    @everywhere g4f6=slce2($z6v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    @everywhere h4f4=slce2($w4v)
    @everywhere h4f5=slce2($w5v)
    @everywhere h4f6=slce2($w6v)
    nn=0
    jn=4
    js1=jn
    js2=jn
    js3=jn
    js4=jn
    js5=jn
    js6=jn
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
    g4tf4=transpose(conj(g4f4))
    g4tf5=transpose(conj(g4f5))
    g4tf6=transpose(conj(g4f6))
    h4tf1=transpose(conj(h4f1))
    h4tf2=transpose(conj(h4f2))
    h4tf3=transpose(conj(h4f3))
    h4tf4=transpose(conj(h4f4))
    h4tf5=transpose(conj(h4f5))
    h4tf6=transpose(conj(h4f6))
#     f=(x)->itfro6([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4tf5,g4tf6,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^6
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1)*(js5+1)*(js6+1),4)
    @time p6(js1,js2,js3,js4,js5,js6,t1,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,h4f1,h4f2,h4f3,h4f4,h4f5,h4f6,jm,aj1,5/t1,100,at1,jm61,jm62,B1,B2)
    res1[1,l1+1+ii,l2+1+jj]=t1
    res1[2,l1+1+ii,l2+1+jj]=sum(at1[:,1])/sqrt(sum(at1[:,3])*sum(at1[:,4]))
    res1[3,l1+1+ii,l2+1+jj]=sum(at1[:,2])/sqrt(sum(at1[:,3])*sum(at1[:,4]))
    res1[4,l1+1+ii,l2+1+jj]=sum(at1[:,3])
    res1[5,l1+1+ii,l2+1+jj]=sum(at1[:,4])
    res1[6,l1+1+ii,l2+1+jj]=xy
    res1[7,l1+1+ii,l2+1+jj]=z
#     print("{",res1[1,i],",",res1[2,i],",",res1[3,i],"},")
    end
end
print("{")
for j in 1:size(res1,2)
    for k in 1:size(res1,3)
        print("{",abs(res1[1,j,k]),",",abs(res1[2,j,k]),",",abs(res1[3,j,k]),",",abs(res1[4,j,k]),",",abs(res1[5,j,k]),",",real(res1[6,j,k]),",",real(res1[7,j,k]),"},")
    end
end
print("}")

In [ ]:
l1=0
l2=0
res1=Array{ComplexF64,3}(undef,7,2*l1+1,2*l2+1)
for ii in -l1:l1
    for jj in -l2:l2
    @everywhere t1=2
    jp6=2.0*[1 1 1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    xy=ii/30
    z=1-abs(jj/30)
    z1v=im*[0.0 0.0 jp6[1]]
    z4v=im*[0.0 0.0 -jp6[1]]
    z2v=im*[-sqrt(1-z^2) 0.0 -z]*jp6[1]
    z5v=im*[sqrt(1-z^2) 0.0 z]*jp6[1]
    z3v=im*[xy*sqrt(1-z^2) sqrt(1-xy^2)*sqrt(1-z^2) -z]*jp6[1]
    z6v=im*[-xy*sqrt(1-z^2) -sqrt(1-xy^2)*sqrt(1-z^2) z]*jp6[1]
    w1v=im*[0.0 0.0 jp6[1]]
    w4v=im*[0.0 0.0 -jp6[1]]
    w2v=im*[-sqrt(1-z^2) 0.0 -z]*jp6[1]
    w5v=im*[sqrt(1-z^2) 0.0 z]*jp6[1]
    w3v=im*[xy*sqrt(1-z^2) sqrt(1-xy^2)*sqrt(1-z^2) -z]*jp6[1]
    w6v=im*[-xy*sqrt(1-z^2) -sqrt(1-xy^2)*sqrt(1-z^2) z]*jp6[1]
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere g4f4=slce2($z4v)
    @everywhere g4f5=slce2($z5v)
    @everywhere g4f6=slce2($z6v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    @everywhere h4f4=slce2($w4v)
    @everywhere h4f5=slce2($w5v)
    @everywhere h4f6=slce2($w6v)
    nn=0
    jn=5
    js1=jn
    js2=jn
    js3=jn
    js4=jn
    js5=jn
    js6=jn
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
    g4tf4=transpose(conj(g4f4))
    g4tf5=transpose(conj(g4f5))
    g4tf6=transpose(conj(g4f6))
    h4tf1=transpose(conj(h4f1))
    h4tf2=transpose(conj(h4f2))
    h4tf3=transpose(conj(h4f3))
    h4tf4=transpose(conj(h4f4))
    h4tf5=transpose(conj(h4f5))
    h4tf6=transpose(conj(h4f6))
#     f=(x)->itfro6([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4tf5,g4tf6,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,nn,t1,k)
#     (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
#     val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^6
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1)*(js5+1)*(js6+1),4)
    @time p6(js1,js2,js3,js4,js5,js6,t1,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,h4f1,h4f2,h4f3,h4f4,h4f5,h4f6,jm,aj1,5/t1,100,at1,jm61,jm62,B1,B2)
    res1[1,l1+1+ii,l2+1+jj]=t1
    res1[2,l1+1+ii,l2+1+jj]=sum(at1[:,1])/sqrt(sum(at1[:,3])*sum(at1[:,4]))
    res1[3,l1+1+ii,l2+1+jj]=sum(at1[:,2])/sqrt(sum(at1[:,3])*sum(at1[:,4]))
    res1[4,l1+1+ii,l2+1+jj]=sum(at1[:,3])
    res1[5,l1+1+ii,l2+1+jj]=sum(at1[:,4])
    res1[6,l1+1+ii,l2+1+jj]=xy
    res1[7,l1+1+ii,l2+1+jj]=z
#     print("{",res1[1,i],",",res1[2,i],",",res1[3,i],"},")
    end
end
print("{")
for j in 1:size(res1,2)
    for k in 1:size(res1,3)
        print("{",abs(res1[1,j,k]),",",abs(res1[2,j,k]),",",abs(res1[3,j,k]),",",abs(res1[4,j,k]),",",abs(res1[5,j,k]),",",res1[6,j,k],",",res1[7,j,k],"},")
    end
end
print("}")

In [7]:
l1=0
l2=0
res1=Array{ComplexF64,3}(undef,7,2*l1+1,2*l2+1)
for ii in -l1:-l1
    for jj in l2:l2
    #     ai1=i-1
        @everywhere t1=0.5
#         println("t1=",t1)
        A=pi - asin(2*sqrt(2)/3);
        A1=A+ii/15
        A2=A+jj/15
        jp4=2*[1 1 1 1]
        jp42=2*[1 1 1 1]
        # z1v=im*pv(jp4,A)[:,1]
        # z2v=im*pv(jp4,A)[:,2]
        # z3v=im*pv(jp4,A)[:,3]
        # z4v=im*pv(jp4,A)[:,4]
        if typeof(pv2(jp4,A1,A2)) != String
            z1v=im*pv2(jp4,A1,A2)[1][:,1]
            z2v=im*pv2(jp4,A1,A2)[1][:,2]
            z3v=im*pv2(jp4,A1,A2)[1][:,3]
            z4v=im*pv2(jp4,A1,A2)[1][:,4]
            w1v=im*pv2(jp4,A1,A2)[1][:,1]
            w2v=im*pv2(jp4,A1,A2)[1][:,2]
            w3v=im*pv2(jp4,A1,A2)[1][:,3]
            w4v=im*pv2(jp4,A1,A2)[1][:,4]
        end
        @everywhere g4f1=slce2($z1v)
        @everywhere g4f2=slce2($z2v)
        @everywhere g4f3=slce2($z3v)
        @everywhere g4f4=slce2($z4v)
        @everywhere h4f1=slce2($w1v)
        @everywhere h4f2=slce2($w2v)
        @everywhere h4f3=slce2($w3v)
        @everywhere h4f4=slce2($w4v)
        nn=0
        jn=12
        js1=jn
        js2=jn
        js3=jn
        js4=jn
        k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
        g4tf1=transpose(conj(g4f1))
        g4tf2=transpose(conj(g4f2))
        g4tf3=transpose(conj(g4f3))
        g4tf4=transpose(conj(g4f4))
        h4tf1=transpose(conj(h4f1))
        h4tf2=transpose(conj(h4f2))
        h4tf3=transpose(conj(h4f3))
        h4tf4=transpose(conj(h4f4))
        fr1=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
        fr2=(x)->itfro([x[1] x[2] x[3]],h4tf1,h4tf2,h4tf3,h4tf4,h4f1,h4f2,h4f3,h4f4,nn,t1,k)
        (val,err) = hcubature(fr1, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
        (val2,err2) = hcubature(fr2, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
        val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
        val22=val2*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
        at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),4)
        println(z1v)
        println(z2v)
        println(z3v)
        println(z4v)
        println(g4tf1)
        println(g4tf2)
        println(g4tf3)
        println(g4tf4)
        print(val1)
#         g4tf1=transpose(conj(g4f1))
#         exp(-t*(lb1(jn)))*conj(glc(jn,g4tf1))
#         @time p123(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,h4f1,h4f2,h4f3,h4f4,jm,aj1,2,1000,at1)
#         res1[1,ii,jj]=t1
#         res1[2,ii,jj]=sum(at1[:,1])/sqrt(val1*val22)
#         res1[3,ii,jj]=sum(at1[:,2])/sqrt(val1*val22)
#         res1[4,ii,jj]=sum(at1[:,3])
#         res1[5,ii,jj]=sum(at1[:,4])
#         res1[6,ii,jj]=ii
#         res1[7,ii,jj]=jj
        #print(at1[:,1])
    end
end
# print("{")
# for j in 1:size(res1,2)
#     for k in 1:size(res1,3)
#         print("{",abs(res1[1,j,k]),",",abs(res1[2,j,k]),",",abs(res1[3,j,k]),",",abs(res1[4,j,k]),",",abs(res1[5,j,k]),",",abs(res1[6,j,k]),",",abs(res1[7,j,k]),"},")
#     end
# end
# print("}")

ComplexF64[0.0 + 2.0im, 0.0 + 0.0im, 0.0 + 0.0im]
ComplexF64[-0.0 - 0.6666666666666659im, 0.0 + 1.885618083164127im, 0.0 + 0.0im]
ComplexF64[-0.0 - 0.6666666666666659im, -0.0 - 0.9428090415820641im, -0.0 - 1.6329931618554518im]
ComplexF64[-0.0 - 0.6666666666666683im, -0.0 - 0.9428090415820626im, 0.0 + 1.6329931618554518im]
ComplexF64[1.5430806348152437 - 0.0im -1.1752011936438014 - 0.0im; -1.1752011936438014 - 0.0im 1.5430806348152437 - 0.0im]
ComplexF64[1.5430806348152437 - 0.0im 0.3917337312146 + 1.1079903110454095im; 0.3917337312146 - 1.1079903110454095im 1.5430806348152437 - 0.0im]
ComplexF64[2.50262839132759 - 0.0im 0.3917337312146 - 0.5539951555227051im; 0.3917337312146 + 0.5539951555227051im 0.5835328783028976 - 0.0im]
ComplexF64[0.5835328783028976 - 0.0im 0.3917337312146014 - 0.5539951555227042im; 0.3917337312146014 + 0.5539951555227042im 2.50262839132759 - 0.0im]
7.095137242116478e14

In [11]:
jp4=2*[1 1 1 1]
ii=15
A=pi - asin(2*sqrt(2)/3)
A1=A-ii/15
A2=A-ii/15
typeof(pv2(jp4,A1,A2))

String

In [ ]:
l1=15
l2=15
res1=Array{ComplexF64,3}(undef,7,2*l1+1,2*l2+1)
for ii in -l1:l1
    for jj in -l2:l2
    #     ai1=i-1
        @everywhere t1=1
#         println("t1=",t1)
        A=pi - asin(2*sqrt(2)/3);
        jp4=2*[1 1 1 1]+(2/10)*ii*[1 0 0 0]
        jp42=2*[1 1 1 1]+(2/10)*jj*[1 0 0 0]
        # z1v=im*pv(jp4,A)[:,1]
        # z2v=im*pv(jp4,A)[:,2]
        # z3v=im*pv(jp4,A)[:,3]
        # z4v=im*pv(jp4,A)[:,4]
        z1v=im*pv(jp4,A)[:,1]
        z2v=im*pv(jp4,A)[:,2]
        z3v=im*pv(jp4,A)[:,3]
        z4v=im*pv(jp4,A)[:,4]
        w1v=im*pv(jp42,A)[:,1]
        w2v=im*pv(jp42,A)[:,2]
        w3v=im*pv(jp42,A)[:,3]
        w4v=im*pv(jp42,A)[:,4]
        @everywhere g4f1=slce2($z1v)
        @everywhere g4f2=slce2($z2v)
        @everywhere g4f3=slce2($z3v)
        @everywhere g4f4=slce2($z4v)
        @everywhere h4f1=slce2($w1v)
        @everywhere h4f2=slce2($w2v)
        @everywhere h4f3=slce2($w3v)
        @everywhere h4f4=slce2($w4v)
        nn=0
        jn=12
        js1=jn
        js2=jn
        js3=jn
        js4=jn
        k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
        g4tf1=transpose(conj(g4f1))
        g4tf2=transpose(conj(g4f2))
        g4tf3=transpose(conj(g4f3))
        g4tf4=transpose(conj(g4f4))
        h4tf1=transpose(conj(h4f1))
        h4tf2=transpose(conj(h4f2))
        h4tf3=transpose(conj(h4f3))
        h4tf4=transpose(conj(h4f4))
        fr1=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
        fr2=(x)->itfro([x[1] x[2] x[3]],h4tf1,h4tf2,h4tf3,h4tf4,h4f1,h4f2,h4f3,h4f4,nn,t1,k)
        (val,err) = hcubature(fr1, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
        (val2,err2) = hcubature(fr2, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
        val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
        val22=val2*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
        at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),4)
#         g4tf1=transpose(conj(g4f1))
#         exp(-t*(lb1(jn)))*conj(glc(jn,g4tf1))
        @time p123(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,h4f1,h4f2,h4f3,h4f4,jm,aj1,2,1000,at1)
        res1[1,ii,jj]=t1
        res1[2,ii,jj]=sum(at1[:,1])/sqrt(val1*val22)
        res1[3,ii,jj]=sum(at1[:,2])/sqrt(val1*val22)
        res1[4,ii,jj]=sum(at1[:,3])
        res1[5,ii,jj]=sum(at1[:,4])
        res1[6,ii,jj]=ii
        res1[7,ii,jj]=jj
        #print(at1[:,1])
    end
end
print("{")
for j in 1:size(res1,2)
    for k in 1:size(res1,3)
        print("{",abs(res1[1,j,k]),",",abs(res1[2,j,k]),",",abs(res1[3,j,k]),",",abs(res1[4,j,k]),",",abs(res1[5,j,k]),",",abs(res1[6,j,k]),",",abs(res1[7,j,k]),"},")
    end
end
print("}")

In [4]:
l1=15
l2=15
res1=Array{ComplexF64,3}(undef,7,2*l1+1,2*l2+1)
for ii in -l1:-l1
    for jj in l2:l2
    #     ai1=i-1
        @everywhere t1=1
#         println("t1=",t1)
        A=pi - asin(2*sqrt(2)/3);
        A1=A+ii/15
        A2=A+jj/15
        jp4=2*[1 1 1 1]
        jp42=2*[1 1 1 1]
        # z1v=im*pv(jp4,A)[:,1]
        # z2v=im*pv(jp4,A)[:,2]
        # z3v=im*pv(jp4,A)[:,3]
        # z4v=im*pv(jp4,A)[:,4]
        if typeof(pv2(jp4,A1,A2)) != String
            z1v=im*pv2(jp4,A1,A2)[1][:,1]
            z2v=im*pv2(jp4,A1,A2)[1][:,2]
            z3v=im*pv2(jp4,A1,A2)[1][:,3]
            z4v=im*pv2(jp4,A1,A2)[1][:,4]
            println(z1v)
            println(z2v)
            println(z3v)
            println(z4v)
        end
    end
end


ComplexF64[0.0 + 2.0im, 0.0 + 0.0im, 0.0 + 0.0im]
ComplexF64[0.0 + 1.22649136816627im, 0.0 + 1.5797844548588365im, 0.0 + 0.0im]
ComplexF64[-0.0 - 1.9468944426571222im, -0.0 - 0.10846075984699116im, -0.0 - 0.4447901670723251im]
ComplexF64[-0.0 - 1.279596925509148im, -0.0 - 1.4713236950118453im, 0.0 + 0.44479016707232505im]


In [ ]:
l1=15
l2=15
res1=Array{ComplexF64,3}(undef,7,2*l1+1,2*l2+1)
for ii in -l1:l1
    for jj in -l2:l2
    #     ai1=i-1
        @everywhere t1=1
#         println("t1=",t1)
        A=pi - asin(2*sqrt(2)/3);
        A1=A+ii/15
        A2=A+jj/15
        jp4=2*[1 1 1 1]
        jp42=2*[1 1 1 1]
        # z1v=im*pv(jp4,A)[:,1]
        # z2v=im*pv(jp4,A)[:,2]
        # z3v=im*pv(jp4,A)[:,3]
        # z4v=im*pv(jp4,A)[:,4]
        if typeof(pv2(jp4,A1,A2)) != String
            z1v=im*pv2(jp4,A1,A2)[1][:,1]
            z2v=im*pv2(jp4,A1,A2)[1][:,2]
            z3v=im*pv2(jp4,A1,A2)[1][:,3]
            z4v=im*pv2(jp4,A1,A2)[1][:,4]
            w1v=im*pv2(jp4,A1,A2)[1][:,1]
            w2v=im*pv2(jp4,A1,A2)[1][:,2]
            w3v=im*pv2(jp4,A1,A2)[1][:,3]
            w4v=im*pv2(jp4,A1,A2)[1][:,4]
            @everywhere g4f1=slce2($z1v)
            @everywhere g4f2=slce2($z2v)
            @everywhere g4f3=slce2($z3v)
            @everywhere g4f4=slce2($z4v)
            @everywhere h4f1=slce2($w1v)
            @everywhere h4f2=slce2($w2v)
            @everywhere h4f3=slce2($w3v)
            @everywhere h4f4=slce2($w4v)
            nn=0
            jn=9
            js1=jn
            js2=jn
            js3=jn
            js4=jn
            k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
            g4tf1=transpose(conj(g4f1))
            g4tf2=transpose(conj(g4f2))
            g4tf3=transpose(conj(g4f3))
            g4tf4=transpose(conj(g4f4))
            h4tf1=transpose(conj(h4f1))
            h4tf2=transpose(conj(h4f2))
            h4tf3=transpose(conj(h4f3))
            h4tf4=transpose(conj(h4f4))
            fr1=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
            fr2=(x)->itfro([x[1] x[2] x[3]],h4tf1,h4tf2,h4tf3,h4tf4,h4f1,h4f2,h4f3,h4f4,nn,t1,k)
            (val,err) = hcubature(fr1, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
            (val2,err2) = hcubature(fr2, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
            val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
            val22=val2*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
            at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),4)
    #         g4tf1=transpose(conj(g4f1))
    #         exp(-t*(lb1(jn)))*conj(glc(jn,g4tf1))
            @time p123(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,h4f1,h4f2,h4f3,h4f4,jm,aj1,2,1000,at1)
            res1[1,l1+1+ii,l2+1+jj]=t1
            res1[2,l1+1+ii,l2+1+jj]=sum(at1[:,1])/sqrt(sum(at1[:,3])*sum(at1[:,4]))
            res1[3,l1+1+ii,l2+1+jj]=sum(at1[:,2])/sqrt(sum(at1[:,3])*sum(at1[:,4]))
            res1[4,l1+1+ii,l2+1+jj]=sum(at1[:,3])/val1
            res1[5,l1+1+ii,l2+1+jj]=sum(at1[:,4])/val22
            res1[6,l1+1+ii,l2+1+jj]=A1
            res1[7,l1+1+ii,l2+1+jj]=A2
        #print(at1[:,1])
        else
            res1[1,l1+1+ii,l2+1+jj]=t1
            res1[2,l1+1+ii,l2+1+jj]=0
            res1[3,l1+1+ii,l2+1+jj]=0
            res1[4,l1+1+ii,l2+1+jj]=0
            res1[5,l1+1+ii,l2+1+jj]=0
            res1[6,l1+1+ii,l2+1+jj]=A1
            res1[7,l1+1+ii,l2+1+jj]=A2
        end
    end
end
print("{")
for j in 1:size(res1,2)
    for k in 1:size(res1,3)
        print("{",abs(res1[1,j,k]),",",abs(res1[2,j,k]),",",abs(res1[3,j,k]),",",abs(res1[4,j,k]),",",abs(res1[5,j,k]),",",abs(res1[6,j,k]),",",abs(res1[7,j,k]),"},")
    end
end
print("}")

In [8]:
ii=-15
jj=-15
nt=20
res1=Array{ComplexF64,2}(undef,7,nt)
for i in 1:nt
        ai1=i-1
        @everywhere t1=1+0.05*$ai1
#         println("t1=",t1)
        A=pi - asin(2*sqrt(2)/3);
        A1=A+ii/15
        A2=A-ii/15
        B1=A+jj/15
        B2=A-jj/15
        jp4=2*[1 1 1 1]
        jp42=2*[1 1 1 1]
        # z1v=im*pv(jp4,A)[:,1]
        # z2v=im*pv(jp4,A)[:,2]
        # z3v=im*pv(jp4,A)[:,3]
        # z4v=im*pv(jp4,A)[:,4]
        z1v=im*pv2(jp4,A1,A2)[1][:,1]
        z2v=im*pv2(jp4,A1,A2)[1][:,2]
        z3v=im*pv2(jp4,A1,A2)[1][:,3]
        z4v=im*pv2(jp4,A1,A2)[1][:,4]
        w1v=im*pv2(jp42,B1,B2)[1][:,1]
        w2v=im*pv2(jp42,B1,B2)[1][:,2]
        w3v=im*pv2(jp42,B1,B2)[1][:,3]
        w4v=im*pv2(jp42,B1,B2)[1][:,4]
        @everywhere g4f1=slce2($z1v)
        @everywhere g4f2=slce2($z2v)
        @everywhere g4f3=slce2($z3v)
        @everywhere g4f4=slce2($z4v)
        @everywhere h4f1=slce2($w1v)
        @everywhere h4f2=slce2($w2v)
        @everywhere h4f3=slce2($w3v)
        @everywhere h4f4=slce2($w4v)
        nn=0
        jn=10
        js1=jn
        js2=jn
        js3=jn
        js4=jn
        k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
        g4tf1=transpose(conj(g4f1))
        g4tf2=transpose(conj(g4f2))
        g4tf3=transpose(conj(g4f3))
        g4tf4=transpose(conj(g4f4))
        h4tf1=transpose(conj(h4f1))
        h4tf2=transpose(conj(h4f2))
        h4tf3=transpose(conj(h4f3))
        h4tf4=transpose(conj(h4f4))
        fr1=(x)->itfro([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4f1,g4f2,g4f3,g4f4,nn,t1,k)
        fr2=(x)->itfro([x[1] x[2] x[3]],h4tf1,h4tf2,h4tf3,h4tf4,h4f1,h4f2,h4f3,h4f4,nn,t1,k)
        (val,err) = hcubature(fr1, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
        (val2,err2) = hcubature(fr2, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
        val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
        val22=val2*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^4
        at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1),4)
#         g4tf1=transpose(conj(g4f1))
#         exp(-t*(lb1(jn)))*conj(glc(jn,g4tf1))
        @time p123(js1,js2,js3,js4,t1,g4f1,g4f2,g4f3,g4f4,h4f1,h4f2,h4f3,h4f4,jm,aj1,2,1000,at1)
        res1[1,i]=t1
        res1[2,i]=sum(at1[:,1])/sqrt(sum(at1[:,3])*sum(at1[:,4]))
        res1[3,i]=sum(at1[:,2])/sqrt(sum(at1[:,3])*sum(at1[:,4]))
        res1[4,i]=sum(at1[:,3])/val1
        res1[5,i]=sum(at1[:,4])/val22
        res1[6,i]=A1
        res1[7,i]=B1
        #print(at1[:,1])
end
print("{")
for j in 1:size(res1,2)
        print("{",abs(res1[1,j]),",",abs(res1[2,j]),",",abs(res1[3,j]),",",abs(res1[4,j]),",",abs(res1[5,j]),",",abs(res1[6,j]),",",abs(res1[7,j]),"},")
end
print("}")

 21.737689 seconds (89.44 k allocations: 6.612 MiB, 0.21% compilation time)
 21.572192 seconds (16.59 k allocations: 2.377 MiB)
 22.165480 seconds (16.61 k allocations: 2.379 MiB)
 21.315169 seconds (16.58 k allocations: 2.377 MiB)
 20.548728 seconds (16.56 k allocations: 2.377 MiB)
 21.269375 seconds (16.58 k allocations: 2.377 MiB)
 21.068380 seconds (16.57 k allocations: 2.377 MiB)
 22.390643 seconds (16.61 k allocations: 2.378 MiB)
 22.533645 seconds (16.62 k allocations: 2.378 MiB)
 20.521443 seconds (16.55 k allocations: 2.376 MiB)
 22.341346 seconds (16.61 k allocations: 2.379 MiB)
 21.385050 seconds (16.58 k allocations: 2.377 MiB)
 21.143434 seconds (16.57 k allocations: 2.377 MiB)
 21.217856 seconds (16.58 k allocations: 2.377 MiB)
 21.314699 seconds (16.58 k allocations: 2.377 MiB)
 20.795138 seconds (16.56 k allocations: 2.376 MiB)
 20.888380 seconds (16.57 k allocations: 2.377 MiB)
 21.170656 seconds (16.57 k allocations: 2.376 MiB)
 23.173516 seconds (16.64 k allocations:

57027.973032 seconds (3.72 M allocations: 188.877 MiB, 0.00% gc time, 0.00% compilation time)
{{0.35,8.255968134835827,1.0,0.9999338216284533,0.9999338216284533,0.9106332362490182,0.9106332362490182},}

In [ ]:
285.34983150369465

In [2]:
pi - asin(2*sqrt(2)/3)

1.9106332362490182

In [21]:
A=pi - asin(2*sqrt(2)/3)
[[-sin(A)/2,-sqrt(3)sin(A)/2,cos(A)] [sin(A),0,cos(A)] [0,0,1] [-sin(A)/2,sqrt(3)sin(A)/2,cos(A)]]

3×4 Matrix{Float64}:
 -0.471405   0.942809  0.0  -0.471405
 -0.816497   0.0       0.0   0.816497
 -0.333333  -0.333333  1.0  -0.333333

In [12]:
l=1
res1=Array{Float64,2}(undef,3,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=1.2
    jp6=2.0*[1 1 1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[0.0 0.0 jp6[1]]
    z4v=im*[0.0 0.0 -jp6[1]]
    z2v=im*[-sqrt(1-z^2) 0.0 -z]*jp6[1]
    z5v=im*[sqrt(1-z^2) 0.0 z]*jp6[1]
    z3v=im*[xy*sqrt(1-z^2) sqrt(1-xy^2)*sqrt(1-z^2) -z]*jp6[1]
    z6v=im*[-xy*sqrt(1-z^2) -sqrt(1-xy^2)*sqrt(1-z^2) z]*jp6[1]
    w1v=im*[0.0 0.0 jp6[1]]
    w4v=im*[0.0 0.0 -jp6[1]]
    w2v=im*[-sqrt(1-z^2) 0.0 -z]*jp6[1]
    w5v=im*[sqrt(1-z^2) 0.0 z]*jp6[1]
    w3v=im*[xy*sqrt(1-z^2) sqrt(1-xy^2)*sqrt(1-z^2) -z]*jp6[1]
    w6v=im*[-xy*sqrt(1-z^2) -sqrt(1-xy^2)*sqrt(1-z^2) z]*jp6[1]
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere g4f4=slce2($z4v)
    @everywhere g4f5=slce2($z5v)
    @everywhere g4f6=slce2($z6v)
    @everywhere h4f1=slce2($w1v)
    @everywhere h4f2=slce2($w2v)
    @everywhere h4f3=slce2($w3v)
    @everywhere h4f4=slce2($w4v)
    @everywhere h4f5=slce2($w5v)
    @everywhere h4f6=slce2($w6v)
    nn=0
    jn=3
    js1=jn
    js2=jn
    js3=jn
    js4=jn
    js5=jn
    js6=jn
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
    g4tf4=transpose(conj(g4f4))
    g4tf5=transpose(conj(g4f5))
    g4tf6=transpose(conj(g4f6))
    h4tf1=transpose(conj(h4f1))
    h4tf2=transpose(conj(h4f2))
    h4tf3=transpose(conj(h4f3))
    h4tf4=transpose(conj(h4f4))
    h4tf5=transpose(conj(h4f5))
    h4tf6=transpose(conj(h4f6))
    f=(x)->itfro6([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4tf5,g4tf6,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,nn,t1,k)
    (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
    val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^6
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1)*(js4+1)*(js5+1)*(js6+1),2)
    @time p6(js1,js2,js3,js4,js5,js6,t1,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,jm,aj1,5/t1,100,at1,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1])/sum(at1[:,2]))
    res1[3,i]=abs(sum(at1[:,2]))/val1
#     print("{",res1[1,i],",",res1[2,i],",",res1[3,i],"},")
    print(res1[3,i],val1)
end

# h5open("./result.h5", "w") do file
#     write(file, "result", res1)
# end

3482.173606 seconds (1.15 G allocations: 292.852 GiB, 1.94% gc time, 0.24% compilation time)
0.95910522741900994.200524559483335e8

In [12]:
l=50
nn1=20
res1=Array{Float64,2}(undef,4,l)
for i in 1:l
    ai1=i-1
    @everywhere t1=2.7+$ai1*0.05
    jp6=2.0*[1 1 1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[0.0 0.0 jp6[1]]
    z2v=im*[0.0 jp6[1] 0.0]
    z3v=im*[jp6[1] 0.0 0.0]
    z4v=im*[0.0 0.0 -jp6[1]]
    z5v=im*[0.0 -jp6[1] 0.0]
    z6v=im*[-jp6[1] 0.0 0.0]
    @everywhere g4f1=slce2($z1v)
    @everywhere g4f2=slce2($z2v)
    @everywhere g4f3=slce2($z3v)
    @everywhere g4f4=slce2($z4v)
    @everywhere g4f5=slce2($z5v)
    @everywhere g4f6=slce2($z6v)
    nn=0
    jn=4
    js1=jn
    js2=jn
    js3=jn
    js4=jn
    js5=jn
    js6=jn
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
    g4tf4=transpose(conj(g4f4))
    g4tf5=transpose(conj(g4f5))
    g4tf6=transpose(conj(g4f6))
    # f=(x)->itfro6([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4tf5,g4tf6,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,nn,t1,k)
    # (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
    # val1=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^6
    at1=SharedArray{Complex{Float64}}((js1+1)*(js2+1)*(js3+1),nn1)
    @time tran6(js1,js2,js3,js4,js5,js6,t1,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,jm,aj1,5/t1,100,at1,nn1,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1])/sum(at1[:,2]))
    res1[3,i]=abs(sum(at1[:,2]))
    for jj in 4:nn1
        res1[jj,i]=abs(sum(at1[:,jj-1]))
    end
    str1="{"*string(res1[1,i])
    for kk in 2:nn1
        str1=str1*","*string(res1[kk,i])
    end
    str1=str1*"},"
    print(str1)
    # res1[4,i]=abs(sum(at1[:,2]))/val1
#     print("{",res1[1,i],",",res1[2,i],",",res1[3,i],"},")
end

 43.031216 seconds (18.47 k allocations: 3.048 MiB, 0.13% gc time)
 43.643837 seconds (18.47 k allocations: 3.046 MiB)
 43.435780 seconds (18.47 k allocations: 3.048 MiB)
 41.869151 seconds (18.42 k allocations: 3.045 MiB)
 43.772708 seconds (18.47 k allocations: 3.047 MiB)
 45.259702 seconds (18.52 k allocations: 3.048 MiB)
 47.068240 seconds (18.58 k allocations: 3.051 MiB)
 45.654691 seconds (18.53 k allocations: 3.040 MiB)
 45.767626 seconds (18.53 k allocations: 3.038 MiB)
 41.869487 seconds (18.41 k allocations: 3.037 MiB)
 44.178939 seconds (18.49 k allocations: 3.048 MiB)
 44.768279 seconds (18.50 k allocations: 3.035 MiB)
 47.403686 seconds (18.59 k allocations: 3.051 MiB)
 45.603624 seconds (18.52 k allocations: 3.036 MiB)
 49.788777 seconds (18.66 k allocations: 3.053 MiB)
 47.193774 seconds (18.57 k allocations: 3.038 MiB)
 51.401839 seconds (18.70 k allocations: 3.043 MiB)
 51.201458 seconds (18.69 k allocations: 3.041 MiB)
 53.021030 seconds (18.75 k allocations: 3.044 Mi

In [21]:
l=1
res1=SharedArray{Float64,2}(3,l)
val1=[0.0]
@sync @distributed for i in 1:l
    ai1=i-1
    t1=2.88
    jp6=2.0*[1 1 1 1 1 1]
    # z1v=im*pv(jp4,A)[:,1]
    # z2v=im*pv(jp4,A)[:,2]
    # z3v=im*pv(jp4,A)[:,3]
    # z4v=im*pv(jp4,A)[:,4]
    z1v=im*[0.0 0.0 jp6[1]]
    z2v=im*[0.0 jp6[1] 0.0]
    z3v=im*[jp6[1] 0.0 0.0]
    z4v=im*[0.0 0.0 -jp6[1]]
    z5v=im*[0.0 -jp6[1] 0.0]
    z6v=im*[-jp6[1] 0.0 0.0]
    g4f1=slce2(z1v)
    g4f2=slce2(z2v)
    g4f3=slce2(z3v)
    g4f4=slce2(z4v)
    g4f5=slce2(z5v)
    g4f6=slce2(z6v)
    nn=0
    jn=5
    js1=jn
    js2=jn
    js3=jn
    js4=jn
    js5=jn
    js6=jn
    k=[1.0+im*0.0 0.0+im*0.0;0.0+im*0.0 1.0+im*0.0]
    g4tf1=transpose(conj(g4f1))
    g4tf2=transpose(conj(g4f2))
    g4tf3=transpose(conj(g4f3))
    g4tf4=transpose(conj(g4f4))
    g4tf5=transpose(conj(g4f5))
    g4tf6=transpose(conj(g4f6))
    f=(x)->itfro6([x[1] x[2] x[3]],g4tf1,g4tf2,g4tf3,g4tf4,g4tf5,g4tf6,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,nn,t1,k)
    (val,err) = hcubature(f, [0 0 0], [pi pi 2*pi], reltol=1e-6, abstol=1e-6, maxevals=0)
    val1[1]=val*(2*sqrt(pi)*exp(t1/4)/(t1^(3/2)))^6
    at1=Array{Complex{Float64}}(undef,(js1+1)*(js2+1)*(js3+1)*(js4+1)*(js5+1)*(js6+1),2)
    @time tran6(js1,js2,js3,js4,js5,js6,t1,g4f1,g4f2,g4f3,g4f4,g4f5,g4f6,jm,aj1,5/t1,100,at1,jm61,jm62,B1,B2)
    res1[1,i]=abs(t1)
    res1[2,i]=abs(sum(at1[:,1])/sum(at1[:,2]))
    res1[3,i]=abs(sum(at1[:,2]))
#     print("{",res1[1,i],",",res1[2,i],",",res1[3,i],"},")
    print(res1[3,i],val1)
end

# h5open("./result.h5", "w") do file
#     write(file, "result", res1)
# end

      From worker 2:	114.149263 seconds (171.04 M allocations: 30.175 GiB, 2.80% gc time, 4.24% compilation time)


Task (done) @0x00000000cbbd8b00